# Chapter 4 / Paper 3

## Consumer Mobility and the Spatial Organization of Local Market Access

This notebook documents the analytical workflow used in Chapter 4 of the dissertation. It examines whether aggregated consumer mobility reveals recurrent and spatially differentiated access conditions that visit volume alone does not represent.

The workflow evaluates the retained Market Infrastructure Index (MII), estimates localized mobility signatures, identifies exploratory mobility-based access regimes, compares those regimes with VIIRS nighttime-light intensity as bounded external context, and allocates tract-level MII values to a 100 m grid in the Sao Paulo metropolitan window under an explicit aggregate-preserving rule.

> **Interpretation boundary.** Mobility-based market infrastructure is a pre-outcome access dimension, not demand, market opportunity, sales, conversion, or performance. VIIRS provides contextual proxy evidence, and the 100 m surface is model-based rather than observed mobility at that resolution.

The project that originated this workflow received third place in the I-GUIDE Spatial AI Challenge 2025-26. This public copy uses the terminology and interpretation adopted in the dissertation.


# 1. Introduction

Firms often need to compare local markets before reliable sales or other direct performance measures are available, although visit volume alone cannot show whether access to a place is recurrent, broadly distributed among visitors, or stable during the observation period. This chapter examines whether aggregated mobility can make that earlier access structure observable without treating movement as demand or performance.

We use **mobility-based market infrastructure** for the recurrent and temporally stable organization of local access captured through mobility data, treating it as one dimension of the broader latent market infrastructure developed in Chapter 2. Seven tract-level indicators describe visitation scale, visitor reach, inflow, recurrence, duration, and temporal stability, while the retained **Market Infrastructure Index (MII)** combines those indicators as an equal-weight empirical representation of mobility-based access conditions.

The notebook asks whether the retained index follows the dominant covariance structure of the seven indicators, whether that access structure is spatially organized and locally differentiated, whether recurring local configurations appear in different nighttime-light contexts, and what a transparent change in spatial support reveals about variation concealed within census tracts. The analyses remain descriptive and diagnostic throughout.

# 2. Data and Operational Inputs

## 2.1 Analytical inputs and access conditions

The workflow begins from two analytical inputs: a restricted tract-level GeoPackage containing the mobility representation and a locally supplied annual VIIRS nighttime-light archive. The GeoPackage provides census-tract geometry, the seven standardized mobility indicators, and the retained `mii` and `mii_norm` fields, while VIIRS supplies bounded external context for local economic intensity and the ancillary surface used in the Sao Paulo allocation.

The repository does not redistribute the mobility GeoPackage or provide a restricted provider link. Authorized users must place a local copy in the path documented in `data/README.md`, while the VIIRS archive should be obtained from its official provider.

## 2.1b Analysis-ready spatial structure

The restricted GeoPackage contains one row per census tract, a stable tract identifier (`ct_id`), geometry, standardized mobility indicators, and the retained MII fields. This structure supports covariance diagnostics, spatial autocorrelation, localized PCA, sensitivity analyses, clustering, VIIRS comparison, and aggregate-preserving spatial allocation from a common tract-level input.

The MII was created during preprocessing and is not recomputed by PCA in this notebook. PCA is used later to examine whether the equal-weight index aligns with the dominant covariance pattern among its component indicators.

## 2.2 Asset ledger

The table below records the operational role, access status, and expected local location of each input. It distinguishes supplied fields from quantities recomputed in the notebook and does not imply that the restricted mobility data are openly reproducible.

In [ ]:
# ============================================================
# SECTION 2 — Imports, project-relative paths, and asset ledger
# ============================================================

from pathlib import Path
from zipfile import ZipFile
import os
import warnings

import geopandas as gpd
import pandas as pd
import rasterio
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)

chapter_dir = Path.cwd().resolve()
if chapter_dir.name == "notebooks":
    chapter_dir = chapter_dir.parent
if not (chapter_dir / "data").exists():
    raise RuntimeError(
        "Start Jupyter from the chapter_4_paper_3 directory or its notebooks directory."
    )
os.chdir(chapter_dir)

data_dir = chapter_dir / "data"
analysis_zip_path = data_dir / "private" / "mii_brazil_analysis_ready.gpkg.zip"
viirs_zip_path = data_dir / "external" / "VIIRS_annual_2024.zip"
analysis_extract_dir = data_dir / "private" / "analysis_ready_asset"
viirs_extract_dir = data_dir / "external" / "viirs_2024_asset"
outputs_dir = chapter_dir / "outputs"

asset_ledger = pd.DataFrame(
    [
        {
            "asset": "Analysis-ready GeoPackage",
            "access": "Restricted; authorized local copy required",
            "local_file": str(analysis_zip_path),
            "required_contents": (
                "ct_id, geometry, mii, mii_norm, and seven standardized mobility indicators"
            ),
            "analytical_role": (
                "National MII diagnostics, localized signatures, exploratory regimes, "
                "VIIRS joins, and the Sao Paulo allocation base"
            ),
        },
        {
            "asset": "Annual 2024 VIIRS nighttime-light archive",
            "access": "Publicly obtainable from the official provider",
            "local_file": str(viirs_zip_path),
            "required_contents": "Georeferenced annual raster files",
            "analytical_role": (
                "Bounded external context and ancillary allocation surface"
            ),
        },
    ]
)

display(asset_ledger)

## 2.3 Local setup and staging

The next cells import the required libraries, define staging helpers, verify the two locally supplied archives, extract them into project-relative directories, and locate the GeoPackage and VIIRS raster files used by the workflow. No empirical input is downloaded by the notebook, so execution begins only after an authorized user has placed both archives in the locations documented in `data/README.md`.

Cached extraction is used when the local archives have already been staged, which keeps the file logic portable and avoids dependence on undocumented machine-specific paths.


In [ ]:
# ============================================================
# SECTION 2 — Local staging and verification helpers
# ============================================================

def require_local_file(path: Path, description: str) -> Path:
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        raise FileNotFoundError(
            f"{description} was not found at {path}. "
            "See data/README.md for access and placement requirements."
        )
    return path


def unzip_file(zip_path: Path, extract_dir: Path) -> Path:
    zip_path = Path(zip_path)
    extract_dir = Path(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)

    sentinel = extract_dir / ".unzipped"
    if sentinel.exists():
        return extract_dir

    with ZipFile(zip_path, "r") as archive:
        archive.extractall(extract_dir)

    sentinel.touch()
    return extract_dir


def is_valid_staged_file(path: Path) -> bool:
    return "__MACOSX" not in path.parts and not path.name.startswith("._")


def find_single_file(root_dir: Path, pattern: str) -> Path:
    matches = sorted(
        path
        for path in Path(root_dir).rglob(pattern)
        if is_valid_staged_file(path)
    )
    if not matches:
        raise FileNotFoundError(
            f"No valid files matched pattern '{pattern}' in {root_dir}"
        )
    if len(matches) > 1:
        print(f"Multiple valid matches found for {pattern}. Using: {matches[0]}")
    return matches[0]


def summarize_viirs_file(raster_path: Path) -> dict:
    with rasterio.open(raster_path) as src:
        values = src.read(1, masked=True)
        return {
            "file": raster_path.name,
            "shape": src.shape,
            "crs": str(src.crs),
            "bounds": str(src.bounds),
            "min": float(values.min()),
            "max": float(values.max()),
            "mean": float(values.mean()),
        }

In [ ]:
# ============================================================
# SECTION 2 — Verify and extract locally supplied assets
# ============================================================

require_local_file(analysis_zip_path, "Restricted analysis-ready GeoPackage archive")
require_local_file(viirs_zip_path, "Annual 2024 VIIRS archive")

unzip_file(analysis_zip_path, analysis_extract_dir)
unzip_file(viirs_zip_path, viirs_extract_dir)

In [ ]:
# ============================================================
# SECTION 2 — Locate staged files
# ============================================================

analysis_gpkg_path = find_single_file(analysis_extract_dir, "*.gpkg")
viirs_tiles = sorted(
    path for path in viirs_extract_dir.rglob("*.tif")
    if is_valid_staged_file(path)
)

staged_assets = pd.DataFrame(
    [
        {
            "asset": "Analysis-ready GeoPackage",
            "resolved_path": str(analysis_gpkg_path),
            "exists": analysis_gpkg_path.exists(),
            "n_files": 1,
        },
        {
            "asset": "VIIRS annual nighttime lights archive",
            "resolved_path": str(viirs_extract_dir),
            "exists": viirs_extract_dir.exists(),
            "n_files": len(viirs_tiles),
        },
    ]
)

display(staged_assets)

## 2.4 GeoPackage verification

We verified that the analysis-ready GeoPackage could be loaded, preserved one row per census tract, and contained the fields required by the subsequent workflow. The core integrity checks covered `ct_id`, `geometry`, `mii`, and `mii_norm`, while the broader inventory confirmed the availability of the seven standardized indicators used for global and localized covariance analysis.

Passing these checks establishes that an authorized local copy has the expected analytical structure; it does not validate the MII or the theoretical construct represented in the chapter.


In [ ]:
# ============================================================
# SECTION 2.4 — Load and verify the analysis-ready GeoPackage
# ============================================================

if "analysis_gpkg_path" not in globals():
    raise RuntimeError("GeoPackage path is not available. Run the staging cells first.")

print("Resolved GeoPackage path:", analysis_gpkg_path)

if not analysis_gpkg_path.exists():
    raise FileNotFoundError(f"GeoPackage not found: {analysis_gpkg_path}")

gdf = gpd.read_file(analysis_gpkg_path)

required_columns = ["ct_id", "geometry", "mii", "mii_norm"]
missing_required_columns = [col for col in required_columns if col not in gdf.columns]

geo_checks = pd.DataFrame(
    {
        "check": [
            "Rows",
            "Unique ct_id",
            "Duplicated ct_id",
            "CRS",
            "Number of columns",
            "Missing required columns",
            "Missing ct_id",
            "Missing geometry",
            "Missing mii",
            "Missing mii_norm",
        ],
        "value": [
            len(gdf),
            gdf["ct_id"].nunique(),
            int(gdf["ct_id"].duplicated().sum()),
            str(gdf.crs),
            len(gdf.columns),
            "None" if not missing_required_columns else ", ".join(missing_required_columns),
            int(gdf["ct_id"].isna().sum()),
            int(gdf["geometry"].isna().sum()),
            int(gdf["mii"].isna().sum()),
            int(gdf["mii_norm"].isna().sum()),
        ],
    }
)

display(geo_checks)

In [ ]:
# ============================================================
# SECTION 2.4 — Column inventory and key-field preview
# ============================================================

if "gdf" not in globals():
    raise RuntimeError("GeoDataFrame 'gdf' is not available. Run the GeoPackage loading cell first.")

columns_df = pd.DataFrame(
    {
        "column_number": range(1, len(gdf.columns) + 1),
        "column_name": list(gdf.columns),
    }
)

preview_columns = [col for col in ["ct_id", "mii", "mii_norm"] if col in gdf.columns]

display(columns_df)
display(gdf[preview_columns].head())

The checks verify whether the authorized GeoPackage has the one-row-per-tract structure and required fields used in the analysis. In the study environment, the file contained 387,779 census tracts, unique identifiers, valid geometry, the retained MII, its normalized version, and the standardized mobility indicators required for the subsequent workflow.

### 2.4.1 Data dictionary overview

The public data dictionary in `data_dictionary/` documents the fields used in the workflow. The restricted GeoPackage contains additional variables, but the main analytical groups are:

| Field group | Example fields | Role in the workflow |
|---|---|---|
| Spatial identifiers and geometry | `ct_id`, `geometry`, `area_km2` | Identify census tracts and support joins, mapping, and changes in spatial support. |
| Aggregated mobility counts | `visits`, `unique`, `repeat_visitors`, `new_visitors` | Describe tract-level visitation before standardization. |
| Weekly mobility summaries | `unique_q1` to `unique_q4`, `visits_q1` to `visits_q4`, `repeat_q1` to `repeat_q4`, `new_visitor_q1` to `new_visitor_q4` | Capture variation across the four observed weeks. |
| Weekly stability indicators | `unique_week_mean`, `unique_week_cv`, `visits_week_mean`, `visits_week_cv` | Describe regularity or volatility during the observation month. |
| Standardized analytical indicators | `z_log1p_visits_A4`, `z_log1p_unique_A4`, `z_log1p_repeat_visitors_A4`, `z_log1p_new_visitors_A4`, `z_dwell_time_mins`, `z_stability_unique_week_cv_A4`, `z_stability_visits_week_cv_A4` | Supply the seven indicators used for PCA diagnostics, localized covariance analysis, sensitivity checks, and regime discovery. |
| MII fields | `mii`, `mii_norm` | Store the retained equal-weight index and its normalized version. |

Tract-level regime assignments, VIIRS means, and allocation diagnostics are generated later in the notebook.

## 2.5 VIIRS verification

The following checks establish whether the annual 2024 nighttime-light archive is locally available, readable, and geospatially valid. The number of raster files reflects how the archive was packaged for computation and has no substantive interpretation.

In [ ]:
# ============================================================
# SECTION 2.5 — Verify the VIIRS nighttime lights archive
# ============================================================

if "viirs_tiles" not in globals():
    raise RuntimeError("VIIRS tiles are not available. Run the staging cells first.")

sample_tile_summary = summarize_viirs_file(viirs_tiles[0]) if len(viirs_tiles) > 0 else None

viirs_checks = pd.DataFrame(
    [
        {
            "zip_file": viirs_zip_path.name,
            "zip_size_mb": round(viirs_zip_path.stat().st_size / (1024 * 1024), 2),
            "tile_count": len(viirs_tiles),
            "sample_tile": sample_tile_summary["file"] if sample_tile_summary else "None",
            "sample_crs": sample_tile_summary["crs"] if sample_tile_summary else "None",
            "sample_shape": sample_tile_summary["shape"] if sample_tile_summary else "None",
        }
    ]
)

display(viirs_checks)

if sample_tile_summary is not None:
    display(pd.DataFrame([sample_tile_summary]))

The VIIRS check records archive readability, raster metadata, and the files available for processing. VIIRS is used only as contextual proxy evidence and as an ancillary allocation surface; it is not interpreted as sales, demand, conversion, or market performance.

## 2.6 Supplied and recomputed elements

The workflow distinguishes the fields supplied in the analysis-ready assets from the quantities calculated in the notebook.

**Supplied locally**
- tract identifiers and geometry;
- seven standardized mobility indicators;
- retained `mii` and `mii_norm` fields; and
- annual 2024 VIIRS raster files.

**Calculated in the notebook**
- global covariance diagnostics and PC1-MII alignment;
- KNN Moran's I for normalized MII;
- localized covariance signatures and sensitivity checks;
- exploratory mobility-based access regimes;
- tract-level VIIRS means, held-out benchmark comparisons, and regime-specific spline profiles; and
- São Paulo aggregate-preserving allocation and within-tract allocation-dispersion diagnostics.

This separation documents the analytical starting point without presenting the restricted inputs as public data or implying that PCA created the retained MII.


## 2.7 Transition to analysis

The staged GeoPackage supplies the tract-level mobility indicators and retained MII fields used in the national workflow, while the VIIRS archive supplies the contextual raster required for the external comparison and the ancillary weights used in the São Paulo allocation. The next section examines the covariance structure of the seven standardized indicators and compares that structure with the MII, which had already been constructed during preprocessing.


# 3. Global Covariance Alignment and Spatial Organization of the MII

The MII had already been constructed during preprocessing as the equal-weight arithmetic mean of seven nationally standardized mobility indicators. This section uses PCA to describe the covariance structure of those indicators and to examine whether the retained index follows their dominant component, after which KNN Moran's I evaluates whether normalized MII values are spatially organized under an explicit neighborhood definition.

These checks assess internal alignment and global spatial dependence. They do not create the MII, establish reliability, completely validate mobility-based market infrastructure, or convert visit volume and the other indicators into measures of demand or market performance.


In [ ]:
# ============================================================
# SECTION 3 — Imports and mobility indicator configuration
# ============================================================

# If needed in a clean environment, uncomment the next line:

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid", context="notebook")

mobility_vars = [
    "z_log1p_visits_A4",
    "z_log1p_unique_A4",
    "z_log1p_new_visitors_A4",
    "z_log1p_repeat_visitors_A4",
    "z_dwell_time_mins",
    "z_stability_unique_week_cv_A4",
    "z_stability_visits_week_cv_A4",
]

mobility_labels = {
    "z_log1p_visits_A4": "Visit volume",
    "z_log1p_unique_A4": "Unique visitors",
    "z_log1p_new_visitors_A4": "New visitors",
    "z_log1p_repeat_visitors_A4": "Repeat visitors",
    "z_dwell_time_mins": "Average dwell time",
    "z_stability_unique_week_cv_A4": "Temporal stability in unique visitors",
    "z_stability_visits_week_cv_A4": "Temporal stability in visit counts",
}

missing_mobility_vars = [col for col in mobility_vars if col not in gdf.columns]
if missing_mobility_vars:
    raise ValueError(
        "The analysis-ready GeoPackage is missing required mobility fields: "
        + ", ".join(missing_mobility_vars)
    )

### 3.1 Mobility signals used in the framework

We characterized mobility behavior with seven standardized indicators capturing complementary dimensions of visitation activity: visit volume, unique visitor presence, inflow of new visitors, repeat-visitation intensity, average dwell time, temporal stability in unique visitors, and temporal stability in visit counts.

Because the shared GeoPackage already contained these fields in standardized form, we used them directly rather than reconstructing transformations outside the notebook.

In [ ]:
# ============================================================
# SECTION 3.1 — Availability and descriptive summary
# ============================================================

mobility_availability = pd.DataFrame(
    {
        "indicator": [mobility_labels[var] for var in mobility_vars],
        "column": mobility_vars,
        "missing_values": [int(gdf[var].isna().sum()) for var in mobility_vars],
        "mean": [round(gdf[var].mean(), 3) for var in mobility_vars],
        "std": [round(gdf[var].std(), 3) for var in mobility_vars],
        "min": [round(gdf[var].min(), 3) for var in mobility_vars],
        "max": [round(gdf[var].max(), 3) for var in mobility_vars],
    }
)

display(mobility_availability)

The descriptive summary records completeness and variation across the seven standardized indicators. Coherence among the indicators is examined in the correlation and PCA diagnostics that follow rather than inferred from their marginal distributions alone.


### 3.2 Correlation structure of the mobility indicators

Pairwise correlations provide an initial view of which indicators vary together and where their relationships diverge. This step precedes PCA and does not assume that all seven measures are interchangeable manifestations of one construct.


In [ ]:
# ============================================================
# SECTION 3.2 — Correlation structure
# ============================================================

corr = gdf[mobility_vars].corr()
corr_labeled = corr.rename(index=mobility_labels, columns=mobility_labels)

plt.figure(figsize=(9, 7))
sns.heatmap(
    corr_labeled,
    annot=True,
    cmap="RdBu_r",
    center=0,
    fmt=".2f",
    square=True,
    linewidths=0.5,
)
plt.title("Correlation Structure of Standardized Mobility Indicators")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

The correlation matrix shows that several count- and stability-related indicators vary together, while average dwell time follows a different pattern. PCA is used next to summarize the shared covariance structure without treating these relationships as complete construct validation.


### 3.3 Dominant covariance dimensions

We fitted PCA to the complete 387,779 by 7 matrix of standardized mobility indicators and inspected the explained-variance shares to determine how much of their common variation was represented by the leading components.


In [ ]:
# ============================================================
# SECTION 3.3 — Principal component analysis
# ============================================================

X_pca = gdf[mobility_vars].dropna().copy()

pca = PCA()
pca.fit(X_pca.values)

# PCA component signs are arbitrary across computational environments.
# We orient PC1 so that it is positively aligned with the stored MII.
initial_pc1_scores = pca.transform(X_pca.values)[:, 0]
stored_mii_for_orientation = gdf.loc[X_pca.index, "mii"].values

initial_alignment = float(
    np.corrcoef(initial_pc1_scores, stored_mii_for_orientation)[0, 1]
)

pc1_orientation = 1.0 if initial_alignment >= 0 else -1.0

if pc1_orientation < 0:
    pca.components_[0] *= -1

explained_variance_df = pd.DataFrame(
    {
        "component": [f"PC{i}" for i in range(1, len(mobility_vars) + 1)],
        "explained_variance_ratio": pca.explained_variance_ratio_,
        "cumulative_explained_variance": np.cumsum(pca.explained_variance_ratio_),
    }
).round(4)

display(explained_variance_df)

plt.figure(figsize=(7.5, 4.5))
plt.plot(
    range(1, len(pca.explained_variance_ratio_) + 1),
    pca.explained_variance_ratio_,
    marker="o",
    linewidth=2,
)
plt.xticks(range(1, len(pca.explained_variance_ratio_) + 1))
plt.xlabel("Principal component")
plt.ylabel("Explained variance ratio")
plt.title("Explained Variance of Mobility Indicators")
plt.tight_layout()
plt.show()

The explained-variance profile shows how concentrated the covariance structure is across components. A large first share indicates a dominant common dimension, while the remaining shares show how much variation still lies outside that dimension.


### 3.4 Interpreting the first principal component

We examined the PC1 loadings to identify which mobility indicators contributed most strongly to the dominant covariance dimension and whether that dimension could be reduced to visit volume alone.


In [ ]:
# ============================================================
# SECTION 3.4 — PC1 loadings
# ============================================================

pc1_loadings_df = pd.DataFrame(
    {
        "indicator": [mobility_labels[var] for var in mobility_vars],
        "column": mobility_vars,
        "pc1_loading": pca.components_[0],
        "absolute_loading": np.abs(pca.components_[0]),
    }
).sort_values("pc1_loading", ascending=True)

display(pc1_loadings_df.round(4))

plt.figure(figsize=(8, 5))
plt.barh(pc1_loadings_df["indicator"], pc1_loadings_df["pc1_loading"], color="#4C78A8")
plt.axvline(0, color="black", linewidth=1)
plt.xlabel("PC1 loading")
plt.title("Loadings of Mobility Indicators on the First Principal Component")
plt.tight_layout()
plt.show()

PC1 combined visitation scale with visitor reach, inflow, recurrence, and weekly stability, while average dwell time loaded more weakly and in the opposite direction. The component consequently summarizes a dominant covariance pattern that includes visit volume without being defined by that measure alone.


### 3.5 Alignment between recomputed PC1 and the retained MII

Because the MII is an equal-weight index created before this notebook's PCA, we orient PC1 to correlate positively with the retained index and calculate their tract-level Pearson correlation. The shared component indicators make this an internal covariance-alignment check rather than an independent reliability test.


In [ ]:
# ============================================================
# SECTION 3.5 — Consistency check between recomputed PC1 and stored MII
# ============================================================

pc1_scores = pca.transform(X_pca.values)[:, 0]
stored_mii = gdf.loc[X_pca.index, "mii"].values

alignment_df = pd.DataFrame(
    {
        "recomputed_pc1": pc1_scores,
        "stored_mii": stored_mii,
    }
)

alignment_corr = float(
    np.corrcoef(alignment_df["recomputed_pc1"], alignment_df["stored_mii"])[0, 1]
)

alignment_summary = pd.DataFrame(
    [
        {
            "observations_used": len(alignment_df),
            "correlation_between_recomputed_pc1_and_stored_mii": round(alignment_corr, 4),
        }
    ]
)

display(alignment_summary)

plt.figure(figsize=(6.5, 5.5))
sns.regplot(
    data=alignment_df,
    x="recomputed_pc1",
    y="stored_mii",
    scatter_kws={"alpha": 0.25, "s": 14},
    line_kws={"color": "darkred", "linewidth": 2},
)
plt.title("Alignment Between Recomputed PC1 and Stored MII")
plt.xlabel("Recomputed first principal component")
plt.ylabel("Stored MII")
plt.tight_layout()
plt.show()

Across 387,779 census tracts, recomputed PC1 correlated `0.9447` with the retained MII. The close alignment shows that the equal-weight index follows the dominant covariance pattern among its component indicators, although the common inputs prevent this result from serving as independent validation of the index or the theoretical construct.

The next diagnostic examines whether normalized MII values are spatially organized under the eight-nearest-neighbor specification used in the chapter.


### 3.6 Global spatial autocorrelation of normalized MII

We calculated global Moran's I for standardized `mii_norm` values using row-standardized eight-nearest-neighbor weights based on tract representative points, with 99 random permutations and seed 42. The statistic evaluates global spatial dependence under this neighborhood definition; it does not identify local clusters, represent polygon contiguity, or establish a mechanism through which access conditions spread between tracts.


In [ ]:
# ============================================================
# SECTION 3.6 — Formal K-nearest-neighbor Moran's I for the MII
# ============================================================

import warnings

try:
    from libpysal.weights import KNN
    from esda.moran import Moran
    moran_dependencies_available = True
except ImportError as exc:
    moran_dependencies_available = False
    moran_import_error = str(exc)

if not moran_dependencies_available:
    moran_knn_summary = pd.DataFrame(
        [
            {
                "status": "not_computed",
                "reason": (
                    "Formal Moran's I requires libpysal and esda. "
                    f"Import error: {moran_import_error}"
                ),
            }
        ]
    )
    display(moran_knn_summary)

else:
    moran_field = "mii_norm"
    moran_k_neighbors = 8
    moran_permutations = 99

    if "gdf" not in globals():
        raise RuntimeError("GeoDataFrame 'gdf' is not available. Run Section 2.4 first.")

    if moran_field not in gdf.columns:
        raise RuntimeError(f"Field '{moran_field}' is not available in the GeoDataFrame.")

    valid_moran_mask = gdf[moran_field].notna() & gdf.geometry.notna()

    gdf_moran = gdf.loc[
        valid_moran_mask,
        ["ct_id", moran_field, "geometry"],
    ].copy()

    if len(gdf_moran) <= moran_k_neighbors:
        raise RuntimeError(
            f"Not enough valid observations to compute KNN Moran's I with k = {moran_k_neighbors}."
        )

    representative_points = gdf_moran.geometry.representative_point()
    lat = np.deg2rad(representative_points.y.to_numpy(dtype="float64"))
    lon = np.deg2rad(representative_points.x.to_numpy(dtype="float64"))

    earth_radius_m = 6_371_000.0
    mean_lat = float(np.nanmean(lat))

    coords_moran = np.column_stack(
        [
            earth_radius_m * lon * np.cos(mean_lat),
            earth_radius_m * lat,
        ]
    )

    moran_values = gdf_moran[moran_field].to_numpy(dtype="float64")
    moran_values_z = (moran_values - np.nanmean(moran_values)) / np.nanstd(moran_values)

    np.random.seed(42)

    # Suppress the expected disconnected-component warning from national KNN weights.
    # The number of components is reported explicitly in the summary table below.
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="The weights matrix is not fully connected*",
            category=UserWarning,
        )
        knn_weights = KNN.from_array(
            coords_moran,
            k=moran_k_neighbors,
        )

    knn_weights.transform = "R"

    moran_result = Moran(
        moran_values_z,
        knn_weights,
        permutations=moran_permutations,
    )

    spatial_lag_mii_z = knn_weights.sparse.dot(moran_values_z)

    moran_knn_summary = pd.DataFrame(
        [
            {
                "field": moran_field,
                "variable_analyzed": "Normalized MII, standardized before Moran's I",
                "tracts_used": len(gdf_moran),
                "weights": f"K-nearest neighbors, k = {moran_k_neighbors}",
                "weight_transform": "row-standardized",
                "connected_components": getattr(knn_weights, "n_components", "not_reported"),
                "permutations": moran_permutations,
                "moran_i": round(float(moran_result.I), 4),
                "expected_i": round(float(moran_result.EI), 6),
                "z_norm": round(float(moran_result.z_norm), 4),
                "p_norm": round(float(moran_result.p_norm), 6),
                "p_sim": round(float(moran_result.p_sim), 6),
            }
        ]
    )

    display(moran_knn_summary)

    moran_plot_df = pd.DataFrame(
        {
            "standardized_mii": moran_values_z,
            "spatial_lag_standardized_mii": spatial_lag_mii_z,
        }
    ).sample(
        n=min(10000, len(gdf_moran)),
        random_state=42,
    )

    plt.figure(figsize=(7.2, 6))
    sns.regplot(
        data=moran_plot_df,
        x="standardized_mii",
        y="spatial_lag_standardized_mii",
        scatter_kws={"alpha": 0.22, "s": 10, "color": "#4C78A8"},
        line_kws={"color": "#8B1A1A", "linewidth": 2},
    )
    plt.axhline(0, color="#777777", linewidth=0.8, linestyle="--")
    plt.axvline(0, color="#777777", linewidth=0.8, linestyle="--")
    plt.title("Moran Scatterplot for the Normalized MII")
    plt.xlabel("Standardized normalized MII")
    plt.ylabel("Spatial lag of standardized normalized MII")
    plt.text(
        0.03,
        0.95,
        f"KNN Moran's I = {moran_result.I:.4f}",
        transform=plt.gca().transAxes,
        ha="left",
        va="top",
        fontsize=11,
        bbox=dict(facecolor="white", edgecolor="#d0d0d0", boxstyle="round,pad=0.35"),
    )
    plt.tight_layout()
    plt.show()

Global Moran's I was `0.4550`, compared with an expected value of `-0.000003` under spatial randomness, and the permutation result reached the smallest attainable probability with 99 permutations (`p_sim = .01`). Under the eight-nearest-neighbor definition, relatively high MII values tended to occur near other high values and low values near other low values.

This result documents global spatial organization without locating individual clusters or demonstrating spillovers. The KNN graph contained multiple connected components, so the statistic remains specific to the row-standardized nearest-neighbor structure used here.


# 4. Localized Mobility Signatures and Exploratory Access Regimes

The global PCA identifies a dominant national covariance dimension but cannot show whether the indicators combine in the same way throughout Brazil. We consequently estimated local covariance structures from the same seven standardized indicators at 7,398 spatially distributed focal tracts, using 200 nearest neighbors as the primary neighborhood definition and retaining local PC1 and PC2 scores together with their explained-variance shares.

Distance-weighted GW-PCA and alternative neighborhoods of 100 and 300 tracts were used to examine the sensitivity of these signatures. We then applied k-means to the focal PC1 and PC2 scores and propagated the four focal classifications to the nearest valid census tracts, treating the resulting mobility-based access regimes as exploratory configurations rather than natural market types, rankings, consumer segments, or performance classes.


In [ ]:
# ============================================================
# SECTION 4 — Imports and configuration
# ============================================================

# If needed in a clean environment, uncomment the next line:

from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA

if "mobility_vars" not in globals():
    mobility_vars = [
        "z_log1p_visits_A4",
        "z_log1p_unique_A4",
        "z_log1p_new_visitors_A4",
        "z_log1p_repeat_visitors_A4",
        "z_dwell_time_mins",
        "z_stability_unique_week_cv_A4",
        "z_stability_visits_week_cv_A4",
    ]

if "mobility_labels" not in globals():
    mobility_labels = {
        "z_log1p_visits_A4": "Visit volume",
        "z_log1p_unique_A4": "Unique visitors",
        "z_log1p_new_visitors_A4": "New visitors",
        "z_log1p_repeat_visitors_A4": "Repeat visitors",
        "z_dwell_time_mins": "Average dwell time",
        "z_stability_unique_week_cv_A4": "Temporal stability in unique visitors",
        "z_stability_visits_week_cv_A4": "Temporal stability in visit counts",
    }

regime_vars = mobility_vars.copy()

metric_crs = 5880
target_focal_locations = 25000
k_neighbors = 200
n_regimes = 4
random_state = 42

### 4.1 Preparing the localized workflow

Deterministic grid thinning selects 7,398 focal tracts that provide broad national coverage without constituting a probability sample. Local covariance structures are estimated at these locations and the resulting regime labels are later assigned to the nearest valid tract, which extends the focal classification without fitting a new local PCA for every census tract.


In [ ]:
# ============================================================
# SECTION 4.1 — Prepare valid tract-level inputs for regime discovery
# Memory-aware centroid-coordinate version
# ============================================================

if "gdf" not in globals():
    raise RuntimeError(
        "GeoDataFrame 'gdf' is not available. Run Section 2.4 first to load the analysis-ready GeoPackage."
    )

if "regime_vars" not in globals():
    raise RuntimeError(
        "Regime variables are not available. Run the Section 4 configuration cell first."
    )

missing_regime_vars = [col for col in regime_vars if col not in gdf.columns]
if missing_regime_vars:
    raise ValueError(
        "The GeoPackage is missing required regime fields: "
        + ", ".join(missing_regime_vars)
    )

valid_regime_mask = gdf[regime_vars].notna().all(axis=1)

# Keep only the columns needed for regime modeling.
gdf_regime_valid = gdf.loc[
    valid_regime_mask,
    ["ct_id"] + regime_vars + ["geometry"],
].copy()

# Memory-aware centroid extraction.
# We compute lightweight geographic representative points and convert them
# to approximate metric coordinates without reprojecting all tract polygons.
representative_points = gdf_regime_valid.geometry.representative_point()

lon = representative_points.x.to_numpy(dtype="float64")
lat = representative_points.y.to_numpy(dtype="float64")

# Equirectangular metric approximation centered on Brazil.
# This is used only for nearest-neighbor regime discovery, not for area measurement.
earth_radius_m = 6_371_000
lat0 = np.deg2rad(np.nanmean(lat))

x_metric = earth_radius_m * np.deg2rad(lon) * np.cos(lat0)
y_metric = earth_radius_m * np.deg2rad(lat)

coords_valid = np.column_stack([x_metric, y_metric])

X_valid = gdf_regime_valid[regime_vars].to_numpy(dtype="float64")

regime_prep_summary = pd.DataFrame(
    [
        {
            "total_tracts": len(gdf),
            "valid_tracts_for_regime_analysis": len(gdf_regime_valid),
            "excluded_due_to_missing_regime_fields": int((~valid_regime_mask).sum()),
            "metric_crs": "Approximate equirectangular meters from tract representative points",
            "k_neighbors": k_neighbors,
            "target_focal_locations": target_focal_locations,
            "n_regimes": n_regimes,
        }
    ]
)

display(regime_prep_summary)

At national scale, neighborhood search used representative-point coordinates transformed to approximate metric units, which reduced memory requirements while preserving the nearest-neighbor logic used to estimate localized covariance signatures.


In [ ]:
# ============================================================
# SECTION 4.1 — Select systematically distributed focal locations
# Memory-aware deterministic grid thinning
# ============================================================

def select_focal_locations(coords: np.ndarray, target_n: int) -> np.ndarray:
    """
    Select spatially distributed focal locations using deterministic grid thinning.
    """
    if len(coords) <= target_n:
        return np.arange(len(coords), dtype=np.int64)

    x = coords[:, 0]
    y = coords[:, 1]

    xmin, xmax = float(x.min()), float(x.max())
    ymin, ymax = float(y.min()), float(y.max())

    width = max(xmax - xmin, 1.0)
    height = max(ymax - ymin, 1.0)

    cell_size = max(np.sqrt((width * height) / target_n), 1.0)

    cell_x = np.floor((x - xmin) / cell_size).astype(np.int64)
    cell_y = np.floor((y - ymin) / cell_size).astype(np.int64)

    # Use a structured array to find the first tract in each grid cell
    # without constructing an additional pandas DataFrame.
    grid_keys = np.core.records.fromarrays([cell_x, cell_y], names="cell_x,cell_y")
    _, first_indices = np.unique(grid_keys, return_index=True)

    focal_idx = np.sort(first_indices.astype(np.int64))

    if len(focal_idx) > target_n:
        focal_idx = focal_idx[np.linspace(0, len(focal_idx) - 1, target_n, dtype=int)]

    return np.sort(focal_idx)


focal_idx = select_focal_locations(coords_valid, target_focal_locations)
coords_focal = coords_valid[focal_idx]

focal_summary = pd.DataFrame(
    [
        {
            "candidate_valid_tracts": len(coords_valid),
            "target_focal_locations": target_focal_locations,
            "selected_focal_locations": len(focal_idx),
            "k_neighbors": k_neighbors,
            "n_regimes": n_regimes,
        }
    ]
)

display(focal_summary)

### 4.2 Localized covariance signatures

For each focal tract, we fit a two-component PCA to the seven standardized indicators observed in its 200-neighbor environment, align local component directions with their global counterparts, and retain the focal PC1 and PC2 scores together with the variance shares explained by both components. These values locate each focal tract within its local covariance structure and summarize how strongly the neighborhood is represented by its first two dimensions.


In [ ]:
# ============================================================
# SECTION 4.2 — Estimate localized PCA signatures
# ============================================================

nn_all = NearestNeighbors(
    n_neighbors=k_neighbors,
    algorithm="ball_tree",
)
nn_all.fit(coords_valid)

_, neighbor_indices = nn_all.kneighbors(coords_focal)

global_pc1_vector = np.asarray(pca.components_[0])
global_pc2_vector = np.asarray(pca.components_[1])

local_signature_rows = []

for focal_position, tract_idx in enumerate(focal_idx):
    local_idx = neighbor_indices[focal_position]
    X_local = X_valid[local_idx]

    local_pca = PCA(n_components=2)
    local_pca.fit(X_local)

    local_pc1 = local_pca.components_[0].copy()
    local_pc2 = local_pca.components_[1].copy()

    if np.dot(local_pc1, global_pc1_vector) < 0:
        local_pc1 *= -1

    if np.dot(local_pc2, global_pc2_vector) < 0:
        local_pc2 *= -1

    focal_centered = X_valid[tract_idx] - local_pca.mean_
    local_scores = np.array(
        [
            np.dot(focal_centered, local_pc1),
            np.dot(focal_centered, local_pc2),
        ]
    )

    local_signature_rows.append(
        {
            "ct_id": gdf_regime_valid.iloc[tract_idx]["ct_id"],
            "focal_pc1_score": float(local_scores[0]),
            "focal_pc2_score": float(local_scores[1]),
            "pc1_share": float(local_pca.explained_variance_ratio_[0]),
            "pc2_share": float(local_pca.explained_variance_ratio_[1]),
        }
    )

focal_signatures = pd.DataFrame(local_signature_rows)

signature_summary = pd.DataFrame(
    [
        {
            "successful_local_pca_fits": len(focal_signatures),
            "neighbors_per_focal_location": k_neighbors,
            "selected_focal_locations": len(focal_idx),
        }
    ]
)

display(signature_summary)
display(focal_signatures.head())

The localized scores and variance shares provide a continuous representation of geographic variation in mobility covariance. The following sensitivity checks examine how strongly that representation depends on distance weighting and neighborhood size before any exploratory regime labels are introduced.


### 4.2b Distance-weighted robustness check

The primary local PCA gives equal weight to the 200 tracts in each focal neighborhood. We also fit an adaptive bi-square GW-PCA that gives greater influence to nearby tracts and compare its focal scores and variance shares with the equally weighted estimates, using the comparison to evaluate sensitivity to geographic weighting rather than to replace the primary specification.


In [ ]:
# ============================================================
# SECTION 4.2b — Geographically weighted PCA robustness check
# ============================================================

def adaptive_bisquare_weights(distances: np.ndarray) -> np.ndarray:
    """
    Compute adaptive bi-square kernel weights for one focal neighborhood.
    The bandwidth is the maximum neighbor distance for the focal location.
    """
    distances = np.asarray(distances, dtype="float64")
    bandwidth = float(np.nanmax(distances))

    if not np.isfinite(bandwidth) or bandwidth <= 0:
        return np.ones_like(distances, dtype="float64") / len(distances)

    scaled_distance = distances / bandwidth
    weights = np.where(
        scaled_distance < 1,
        (1 - scaled_distance**2) ** 2,
        0.0,
    )

    if weights.sum() <= 0:
        weights = np.ones_like(distances, dtype="float64")

    return weights / weights.sum()


def weighted_pca_two_components(X_local: np.ndarray, weights: np.ndarray) -> tuple:
    """
    Estimate a two-component PCA from a weighted covariance matrix.
    Returns component vectors, explained variance ratios, and the weighted mean.
    """
    weights = np.asarray(weights, dtype="float64")
    weights = weights / weights.sum()

    weighted_mean = np.average(X_local, axis=0, weights=weights)
    X_centered = X_local - weighted_mean

    weighted_covariance = (X_centered * weights[:, None]).T @ X_centered

    eigenvalues, eigenvectors = np.linalg.eigh(weighted_covariance)
    order = np.argsort(eigenvalues)[::-1]

    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]

    total_variance = eigenvalues.sum()
    explained_share = eigenvalues / total_variance if total_variance > 0 else np.full_like(eigenvalues, np.nan)

    components = eigenvectors[:, :2].T
    explained_ratio = explained_share[:2]

    return components, explained_ratio, weighted_mean


neighbor_distances, gw_neighbor_indices = nn_all.kneighbors(coords_focal)

gw_pca_rows = []

for focal_position, tract_idx in enumerate(focal_idx):
    local_idx = gw_neighbor_indices[focal_position]
    local_distances = neighbor_distances[focal_position]

    X_local = X_valid[local_idx]
    weights = adaptive_bisquare_weights(local_distances)

    gw_components, gw_explained_ratio, gw_weighted_mean = weighted_pca_two_components(
        X_local=X_local,
        weights=weights,
    )

    gw_pc1 = gw_components[0].copy()
    gw_pc2 = gw_components[1].copy()

    # First orient components toward the global PCA axes for consistency.
    if np.dot(gw_pc1, global_pc1_vector) < 0:
        gw_pc1 *= -1

    if np.dot(gw_pc2, global_pc2_vector) < 0:
        gw_pc2 *= -1

    focal_centered = X_valid[tract_idx] - gw_weighted_mean
    gw_scores = np.array(
        [
            np.dot(focal_centered, gw_pc1),
            np.dot(focal_centered, gw_pc2),
        ]
    )

    gw_pca_rows.append(
        {
            "ct_id": gdf_regime_valid.iloc[tract_idx]["ct_id"],
            "gw_pc1_score": float(gw_scores[0]),
            "gw_pc2_score": float(gw_scores[1]),
            "gw_pc1_share": float(gw_explained_ratio[0]),
            "gw_pc2_share": float(gw_explained_ratio[1]),
            "effective_neighbors": int((weights > 0).sum()),
        }
    )

gw_pca_signatures = pd.DataFrame(gw_pca_rows)

gw_pca_comparison = focal_signatures.merge(
    gw_pca_signatures,
    on="ct_id",
    how="inner",
)

# PCA signs are arbitrary. Orient the GW-PCA scores so that they align
# positively with the localized PCA scores used in the main regime workflow.
gw_pc1_score_corr_raw = float(
    gw_pca_comparison["focal_pc1_score"].corr(gw_pca_comparison["gw_pc1_score"])
)
gw_pc2_score_corr_raw = float(
    gw_pca_comparison["focal_pc2_score"].corr(gw_pca_comparison["gw_pc2_score"])
)

if gw_pc1_score_corr_raw < 0:
    gw_pca_comparison["gw_pc1_score"] *= -1
    gw_pca_signatures["gw_pc1_score"] *= -1

if gw_pc2_score_corr_raw < 0:
    gw_pca_comparison["gw_pc2_score"] *= -1
    gw_pca_signatures["gw_pc2_score"] *= -1

gw_pca_robustness_summary = pd.DataFrame(
    [
        {
            "focal_locations_compared": len(gw_pca_comparison),
            "kernel": "adaptive bi-square",
            "neighbors_per_focal_location": k_neighbors,
            "mean_effective_neighbors": round(float(gw_pca_comparison["effective_neighbors"].mean()), 1),
            "pc1_share_correlation": round(
                float(gw_pca_comparison["pc1_share"].corr(gw_pca_comparison["gw_pc1_share"])),
                4,
            ),
            "pc2_share_correlation": round(
                float(gw_pca_comparison["pc2_share"].corr(gw_pca_comparison["gw_pc2_share"])),
                4,
            ),
            "pc1_score_correlation": round(
                float(gw_pca_comparison["focal_pc1_score"].corr(gw_pca_comparison["gw_pc1_score"])),
                4,
            ),
            "pc2_score_correlation": round(
                float(gw_pca_comparison["focal_pc2_score"].corr(gw_pca_comparison["gw_pc2_score"])),
                4,
            ),
        }
    ]
)

display(gw_pca_robustness_summary)
display(gw_pca_comparison.head())

In [ ]:
# ============================================================
# SECTION 4.2b — Visual comparison between localized PCA and GW-PCA
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), facecolor="white")

plot_sample = gw_pca_comparison.sample(
    n=min(5000, len(gw_pca_comparison)),
    random_state=random_state,
)

sns.regplot(
    data=plot_sample,
    x="pc1_share",
    y="gw_pc1_share",
    scatter_kws={"alpha": 0.25, "s": 10, "color": "#4C78A8"},
    line_kws={"color": "#8B1A1A", "linewidth": 2},
    ax=axes[0],
)

axes[0].set_title("Local Explained Variance", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Localized PCA PC1 variance share")
axes[0].set_ylabel("GW-PCA PC1 variance share")
axes[0].grid(True, color="#e6e6e6", linewidth=0.8)

sns.regplot(
    data=plot_sample,
    x="focal_pc1_score",
    y="gw_pc1_score",
    scatter_kws={"alpha": 0.25, "s": 10, "color": "#4C78A8"},
    line_kws={"color": "#8B1A1A", "linewidth": 2},
    ax=axes[1],
)

axes[1].set_title("Local Component Scores", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Localized PCA focal PC1 score")
axes[1].set_ylabel("GW-PCA focal PC1 score")
axes[1].grid(True, color="#e6e6e6", linewidth=0.8)

fig.suptitle(
    "Geographically Weighted PCA Robustness of Local Mobility Structure",
    fontsize=15,
    fontweight="bold",
    y=1.03,
)

plt.tight_layout()
plt.show()

Distance-weighted and equally weighted PC1 scores were closely aligned (`r = 0.9674`), while PC2 alignment was lower but remained positive (`r = 0.7770`). Correlations for the corresponding explained-variance shares were `0.7545` for PC1 and `0.7635` for PC2.

The dominant localized dimension consequently remained similar after distance weighting, whereas the secondary dimension and variance shares were more sensitive. These comparisons delimit the robustness of the local representation and do not make the regimes invariant to every spatial specification.


### 4.2c Sensitivity to neighborhood size

Using a reproducible sample of 5,000 focal tracts, we re-estimated the localized signatures with 100 and 300 nearest neighbors and compared them with the primary 200-neighbor specification. A fixed number of neighbors improves comparability across focal locations but permits the physical extent of neighborhoods to vary with tract density.


In [ ]:
# ============================================================
# SECTION 4.2c — Sensitivity of localized PCA signatures to neighborhood size
# ============================================================

sensitivity_neighbor_sizes = [100, 300]
sensitivity_sample_size = min(5000, len(focal_idx))

rng = np.random.default_rng(random_state)
sensitivity_positions = np.sort(
    rng.choice(
        np.arange(len(focal_idx)),
        size=sensitivity_sample_size,
        replace=False,
    )
)

baseline_sensitivity = focal_signatures.iloc[sensitivity_positions].copy()

sensitivity_rows = []
sensitivity_preview_frames = []

for sensitivity_k in sensitivity_neighbor_sizes:
    nn_sensitivity = NearestNeighbors(
        n_neighbors=sensitivity_k,
        algorithm="ball_tree",
    )
    nn_sensitivity.fit(coords_valid)

    _, sensitivity_neighbor_indices = nn_sensitivity.kneighbors(
        coords_focal[sensitivity_positions]
    )

    local_rows = []

    for local_position, focal_position in enumerate(sensitivity_positions):
        tract_idx = focal_idx[focal_position]
        local_idx = sensitivity_neighbor_indices[local_position]
        X_local = X_valid[local_idx]

        local_pca = PCA(n_components=2)
        local_pca.fit(X_local)

        local_pc1 = local_pca.components_[0].copy()
        local_pc2 = local_pca.components_[1].copy()

        if np.dot(local_pc1, global_pc1_vector) < 0:
            local_pc1 *= -1

        if np.dot(local_pc2, global_pc2_vector) < 0:
            local_pc2 *= -1

        focal_values = X_valid[tract_idx].reshape(1, -1)
        centered_focal_values = focal_values - local_pca.mean_

        focal_pc1_score = float(centered_focal_values @ local_pc1)
        focal_pc2_score = float(centered_focal_values @ local_pc2)

        local_rows.append(
            {
                "ct_id": gdf_regime_valid.iloc[tract_idx]["ct_id"],
                "focal_pc1_score_sensitivity": focal_pc1_score,
                "focal_pc2_score_sensitivity": focal_pc2_score,
                "pc1_share_sensitivity": float(local_pca.explained_variance_ratio_[0]),
                "pc2_share_sensitivity": float(local_pca.explained_variance_ratio_[1]),
            }
        )

    sensitivity_df = pd.DataFrame(local_rows)

    comparison_df = baseline_sensitivity[
        ["ct_id", "focal_pc1_score", "focal_pc2_score", "pc1_share", "pc2_share"]
    ].merge(
        sensitivity_df,
        on="ct_id",
        how="inner",
    )

    # PCA component signs are arbitrary. Align sensitivity scores with the
    # baseline orientation before computing stability diagnostics.
    pc1_corr_raw = comparison_df["focal_pc1_score"].corr(
        comparison_df["focal_pc1_score_sensitivity"]
    )

    if pc1_corr_raw < 0:
        comparison_df["focal_pc1_score_sensitivity"] *= -1

    pc2_corr_raw = comparison_df["focal_pc2_score"].corr(
        comparison_df["focal_pc2_score_sensitivity"]
    )

    if pc2_corr_raw < 0:
        comparison_df["focal_pc2_score_sensitivity"] *= -1

    sensitivity_rows.append(
        {
            "alternative_neighbors": sensitivity_k,
            "focal_locations_compared": len(comparison_df),
            "pc1_score_correlation": round(
                float(comparison_df["focal_pc1_score"].corr(comparison_df["focal_pc1_score_sensitivity"])),
                4,
            ),
            "pc2_score_correlation": round(
                float(comparison_df["focal_pc2_score"].corr(comparison_df["focal_pc2_score_sensitivity"])),
                4,
            ),
            "pc1_share_correlation": round(
                float(comparison_df["pc1_share"].corr(comparison_df["pc1_share_sensitivity"])),
                4,
            ),
            "pc2_share_correlation": round(
                float(comparison_df["pc2_share"].corr(comparison_df["pc2_share_sensitivity"])),
                4,
            ),
        }
    )

    comparison_df["alternative_neighbors"] = sensitivity_k
    sensitivity_preview_frames.append(comparison_df)

localized_pca_sensitivity_summary = pd.DataFrame(sensitivity_rows)

display(localized_pca_sensitivity_summary)

localized_pca_sensitivity_preview = pd.concat(
    sensitivity_preview_frames,
    ignore_index=True,
)

plot_sample = localized_pca_sensitivity_preview.sample(
    n=min(6000, len(localized_pca_sensitivity_preview)),
    random_state=random_state,
)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5), facecolor="white")

sns.scatterplot(
    data=plot_sample,
    x="focal_pc1_score",
    y="focal_pc1_score_sensitivity",
    hue="alternative_neighbors",
    palette="viridis",
    alpha=0.35,
    s=12,
    ax=axes[0],
)

axes[0].set_title("PC1 Score Stability", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Baseline localized PCA score (k = 200)")
axes[0].set_ylabel("Alternative neighborhood score")
axes[0].grid(True, color="#e6e6e6", linewidth=0.8)

sns.scatterplot(
    data=plot_sample,
    x="pc1_share",
    y="pc1_share_sensitivity",
    hue="alternative_neighbors",
    palette="viridis",
    alpha=0.35,
    s=12,
    ax=axes[1],
    legend=False,
)

axes[1].set_title("PC1 Explained-Variance Share Stability", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Baseline PC1 variance share (k = 200)")
axes[1].set_ylabel("Alternative neighborhood PC1 variance share")
axes[1].grid(True, color="#e6e6e6", linewidth=0.8)

handles, labels = axes[0].get_legend_handles_labels()
axes[0].legend(
    handles=handles,
    labels=[f"k = {label}" for label in labels],
    title="Alternative k",
    loc="best",
)

fig.suptitle(
    "Sensitivity of Localized PCA Signatures to Neighborhood Size",
    fontsize=15,
    fontweight="bold",
    y=1.03,
)

plt.tight_layout()
plt.show()

Relative to the 200-neighbor specification, PC1 score correlations reached `0.9833` with 100 neighbors and `0.9937` with 300 neighbors. PC2 score correlations were `0.8106` and `0.8305`, while correlations for PC1 variance shares were `0.7568` and `0.8713` and those for PC2 shares were `0.7447` and `0.8735`.

The leading local score was the most stable feature across neighborhood sizes, while secondary scores and variance shares changed more. The regime analysis that follows consequently partitions a continuous and partly scale-sensitive representation rather than identifying fixed local-market categories.


### 4.3 Identifying exploratory mobility-based access regimes

We applied k-means to focal PC1 and PC2 scores in their original scales, using four clusters, 20 initializations, and seed 42. The four-regime resolution organizes recurring local configurations for mapping and comparison, while the continuous component scores remain the underlying representation and the cluster labels are not interpreted as a unique or scale-invariant taxonomy.


In [ ]:
# ============================================================
# SECTION 4.3 — Cluster localized signatures into regimes
# ============================================================

cluster_features = focal_signatures[["focal_pc1_score", "focal_pc2_score"]].to_numpy()

kmeans = KMeans(
    n_clusters=n_regimes,
    n_init=20,
    random_state=random_state,
)

raw_regimes = kmeans.fit_predict(cluster_features)
focal_signatures["raw_regime"] = raw_regimes

regime_order = (
    focal_signatures.groupby("raw_regime")["pc1_share"]
    .mean()
    .sort_values(ascending=False)
    .index
    .tolist()
)

regime_mapping = {old_label: new_label for new_label, old_label in enumerate(regime_order, start=1)}
focal_signatures["infrastructure_regime"] = focal_signatures["raw_regime"].map(regime_mapping)

regime_counts = (
    focal_signatures["infrastructure_regime"]
    .value_counts()
    .sort_index()
    .rename_axis("infrastructure_regime")
    .reset_index(name="n_focal_locations")
)

display(regime_counts)

In [ ]:
# ============================================================
# SECTION 4.3b — Alternative cluster-count diagnostics
# ============================================================

from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)

cluster_diagnostic_rows = []

for candidate_k in range(2, 9):
    candidate_model = KMeans(
        n_clusters=candidate_k,
        n_init=20,
        random_state=random_state,
    )
    candidate_labels = candidate_model.fit_predict(cluster_features)

    cluster_diagnostic_rows.append(
        {
            "n_clusters": candidate_k,
            "inertia": float(candidate_model.inertia_),
            "silhouette": float(
                silhouette_score(
                    cluster_features,
                    candidate_labels,
                    sample_size=min(5000, len(cluster_features)),
                    random_state=random_state,
                )
            ),
            "calinski_harabasz": float(
                calinski_harabasz_score(cluster_features, candidate_labels)
            ),
            "davies_bouldin": float(
                davies_bouldin_score(cluster_features, candidate_labels)
            ),
        }
    )

cluster_count_diagnostics = pd.DataFrame(cluster_diagnostic_rows).round(4)
display(cluster_count_diagnostics)

In [ ]:
# ============================================================
# SECTION 4.3c — Sensitivity to focal-score scaling
# ============================================================

from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.preprocessing import StandardScaler

standardized_cluster_features = StandardScaler().fit_transform(cluster_features)

standardized_score_model = KMeans(
    n_clusters=n_regimes,
    n_init=20,
    random_state=random_state,
)
standardized_score_labels = standardized_score_model.fit_predict(
    standardized_cluster_features
)

cluster_scale_sensitivity = pd.DataFrame(
    [
        {
            "comparison": "Original PCA-score scales versus standardized focal scores",
            "adjusted_rand_index": adjusted_rand_score(
                raw_regimes,
                standardized_score_labels,
            ),
            "normalized_mutual_information": normalized_mutual_info_score(
                raw_regimes,
                standardized_score_labels,
            ),
        }
    ]
).round(4)

display(cluster_scale_sensitivity)

### 4.4 Assigning focal regime labels to valid census tracts

K-means classifies the 7,398 focal locations, so the national layer is completed by assigning each valid census tract the label of its nearest focal location. This propagation extends the focal classification without re-estimating a local covariance structure at every tract.


In [ ]:
# ============================================================
# SECTION 4.4 — Assign each valid tract to its nearest focal regime
# ============================================================

nn_focal = NearestNeighbors(n_neighbors=1, algorithm="ball_tree")
nn_focal.fit(coords_focal)

_, nearest_focal_idx = nn_focal.kneighbors(coords_valid)
nearest_focal_idx = nearest_focal_idx.flatten()

gdf_regime_valid["focal_pc1_score"] = focal_signatures.iloc[nearest_focal_idx]["focal_pc1_score"].to_numpy()
gdf_regime_valid["focal_pc2_score"] = focal_signatures.iloc[nearest_focal_idx]["focal_pc2_score"].to_numpy()
gdf_regime_valid["pc1_share_local"] = focal_signatures.iloc[nearest_focal_idx]["pc1_share"].to_numpy()
gdf_regime_valid["pc2_share_local"] = focal_signatures.iloc[nearest_focal_idx]["pc2_share"].to_numpy()
gdf_regime_valid["infrastructure_regime"] = focal_signatures.iloc[nearest_focal_idx]["infrastructure_regime"].to_numpy()

regime_output_columns = [
    "ct_id",
    "focal_pc1_score",
    "focal_pc2_score",
    "pc1_share_local",
    "pc2_share_local",
    "infrastructure_regime",
]

gdf = gdf.drop(
    columns=[
        col for col in regime_output_columns[1:]
        if col in gdf.columns
    ],
    errors="ignore",
)

gdf = gdf.merge(
    gdf_regime_valid[regime_output_columns],
    on="ct_id",
    how="left",
)

tract_regime_counts = (
    gdf["infrastructure_regime"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("infrastructure_regime")
    .reset_index(name="n_census_tracts")
)

display(tract_regime_counts)

### 4.5 Profiles of the exploratory regimes

We summarize the tract-level regime assignments through the seven standardized mobility indicators and the propagated local scores and variance shares. These profiles describe recurring configurations of mobility-based access conditions rather than observed market types or levels of economic performance.


In [ ]:
# ============================================================
# SECTION 4.5 — Profiles of the exploratory regimes
# Memory-aware non-geometric summary version
# ============================================================

import gc

required_regime_summary_columns = [
    "infrastructure_regime",
    "pc1_share_local",
    "pc2_share_local",
    "focal_pc1_score",
    "focal_pc2_score",
] + regime_vars

missing_summary_columns = [
    col for col in required_regime_summary_columns
    if col not in gdf.columns
]

if missing_summary_columns:
    raise RuntimeError(
        "The regime summary fields are not available. "
        "Run Sections 4.3 and 4.4 before Section 4.5. "
        "Missing fields: " + ", ".join(missing_summary_columns)
    )

# Release large temporary objects from localized PCA and robustness checks.
# They are no longer needed after tract-level regime labels have been merged into gdf.
temporary_objects_to_release = [
    "neighbor_indices",
    "neighbor_distances",
    "gw_neighbor_indices",
    "nearest_focal_idx",
    "nn_all",
    "nn_focal",
    "nn_sensitivity",
    "gdf_regime_valid",
    "gw_pca_signatures",
    "gw_pca_comparison",
    "gw_pca_rows",
    "sensitivity_preview_frames",
    "localized_pca_sensitivity_preview",
    "plot_sample",
    "comparison_df",
    "sensitivity_df",
    "baseline_sensitivity",
]

for obj_name in temporary_objects_to_release:
    if obj_name in globals():
        del globals()[obj_name]

_ = gc.collect()

# Build a lightweight non-geometric table for summary statistics.
# This avoids carrying census tract geometries into the groupby operation.
regime_summary_df = pd.DataFrame(
    gdf.loc[
        gdf["infrastructure_regime"].notna(),
        required_regime_summary_columns,
    ]
).copy()

regime_summary_df["infrastructure_regime"] = (
    regime_summary_df["infrastructure_regime"].astype(int)
)

regime_profile = (
    regime_summary_df
    .groupby("infrastructure_regime", sort=True)[regime_vars]
    .mean()
    .rename(columns=mobility_labels)
    .round(3)
    .reset_index()
)

display(regime_profile)

regime_signature_summary = (
    regime_summary_df
    .groupby("infrastructure_regime", sort=True)[
        ["pc1_share_local", "pc2_share_local", "focal_pc1_score", "focal_pc2_score"]
    ]
    .mean()
    .round(3)
    .reset_index()
)

display(regime_signature_summary)

del regime_summary_df
_ = gc.collect()

The regime profiles differ in visitation scale, visitor reach, inflow, recurrence, duration, and weekly stability, while their national frequencies are highly unequal. These differences help describe how a common visit-volume signal appears beside other mobility conditions, but they do not establish that one regime represents better access, stronger opportunity, or higher performance.


### 4.6 National distribution of mobility-based access regimes

The map displays where the four propagated labels occur across valid census tracts. It visualizes the exploratory classification without treating boundaries between colors as administrative regions, natural market divisions, or rankings of access quality.


In [ ]:
# ============================================================
# PUBLICATION CARTOGRAPHY HELPERS
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, TwoSlopeNorm, ListedColormap
import numpy as np

plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.titleweight"] = "bold"


def add_north_arrow(ax, x, y, length, color="#2b2b2b"):
    ax.annotate(
        "",
        xy=(x, y + length),
        xytext=(x, y),
        arrowprops=dict(
            facecolor=color,
            edgecolor=color,
            width=2.0,
            headwidth=9,
            headlength=11,
        ),
        zorder=20,
    )
    ax.text(
        x,
        y + length + 0.015 * length,
        "N",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
        color=color,
        zorder=21,
    )


def add_scale_bar(ax, x, y, segment_length_m, n_segments=2, bar_height=10000, text_color="#2b2b2b"):
    for i in range(n_segments):
        face = "#2b2b2b" if i % 2 == 0 else "white"
        ax.add_patch(
            mpatches.Rectangle(
                (x + i * segment_length_m, y),
                segment_length_m,
                bar_height,
                facecolor=face,
                edgecolor="#2b2b2b",
                linewidth=0.6,
                zorder=20,
            )
        )

    ax.text(x, y + 2.2 * bar_height, "0", ha="center", va="bottom", fontsize=8.5, color=text_color)
    ax.text(
        x + segment_length_m,
        y + 2.2 * bar_height,
        f"{int(segment_length_m/1000)}",
        ha="center",
        va="bottom",
        fontsize=8.5,
        color=text_color,
    )
    ax.text(
        x + n_segments * segment_length_m,
        y + 2.2 * bar_height,
        f"{int(n_segments * segment_length_m/1000)} km",
        ha="center",
        va="bottom",
        fontsize=8.5,
        color=text_color,
    )


def finalize_publication_map(
    fig,
    ax,
    title,
    subtitle,
    footnote,
    left=0.08,
    title_y=0.955,
    subtitle_y=0.928,
    footnote_y=0.035,
):
    ax.set_axis_off()

    fig.text(
        left,
        title_y,
        title,
        fontsize=18,
        fontweight="bold",
        ha="left",
        va="top",
    )
    fig.text(
        left,
        subtitle_y,
        subtitle,
        fontsize=11,
        color="#4f4f4f",
        ha="left",
        va="top",
    )
    fig.text(
        left,
        footnote_y,
        footnote,
        fontsize=8.8,
        color="#5a5a5a",
        ha="left",
        va="bottom",
    )

In [ ]:
# ============================================================
# SECTION 4.6 — Display the pre-rendered national regime map
# ============================================================

from IPython.display import Image, display

regime_map_path = outputs_dir / "figure_08_national_mobility_based_access_regimes.png"
if not regime_map_path.exists():
    raise FileNotFoundError(
        f"Pre-rendered map not found: {regime_map_path}. See outputs/README.md."
    )

display(Image(filename=str(regime_map_path)))

The national map is supplied as a pre-rendered PNG because drawing 387,779 census-tract geometries with publication cartography can exceed notebook rendering limits. Its regime assignments come from the tract-level workflow above, while the static file in `outputs/` preserves the exact figure used for inspection.

The map describes the spatial distribution of exploratory mobility-based access regimes. It does not rank markets or identify administrative regions, consumer segments, or performance classes.

# 5. External Contextual Comparison

The exploratory regimes organize localized mobility configurations but do not, by themselves, show whether those configurations occur in different observable economic contexts. We used annual 2024 VIIRS nighttime-light intensity for this bounded comparison, first calculating tract-level means and comparing their distributions among regimes, then estimating a held-out sequence of benchmark models and separate regime-specific spline profiles.

VIIRS is treated only as contextual proxy evidence for local economic intensity. It is not a measure of sales, demand, conversion, profitability, or market performance, and the comparisons below do not validate the MII, establish causal effects, or show geographically independent prediction.


In [ ]:
# ============================================================
# SECTION 5 — Imports and configuration
# ============================================================

# If needed in a clean environment, uncomment the next line:

from rasterstats import zonal_stats
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import SplineTransformer

nightlight_field = "nightlight_mean"
mobility_intensity_var = "z_log1p_visits_A4"
response_random_state = 42

### 5.1 Preparing tract-level nighttime-light extraction

Annual nighttime-light intensity entered this stage as external context rather than as an input to the MII. The following cells retain only the tract identifier and geometry needed for raster extraction, preserve any previously available tract-level VIIRS field, and otherwise calculate the missing tract means from the locally supplied archive.


In [ ]:
# ============================================================
# SECTION 5.1 — Prepare GeoDataFrame for VIIRS extraction
# Memory-aware version: avoid full national reprojection copy
# ============================================================

if "gdf" not in globals():
    raise RuntimeError("GeoDataFrame 'gdf' is not available. Run the earlier sections first.")

if "viirs_tiles" not in globals():
    raise RuntimeError("VIIRS tiles are not available. Run Section 2 first.")

if "nightlight_field" not in globals():
    raise RuntimeError("nightlight_field is not available. Run the Section 5 configuration cell first.")

if gdf.crs is None:
    raise RuntimeError("The tract GeoDataFrame has no CRS. VIIRS extraction requires geographic coordinates.")

# Keep only the fields needed for VIIRS extraction and final merge.
# This avoids copying all national analytical columns.
viirs_columns = ["ct_id", "geometry"]
if nightlight_field in gdf.columns:
    viirs_columns.append(nightlight_field)

gdf_viirs = gdf.loc[:, viirs_columns].copy()

# The shared GeoPackage is expected to be EPSG:4674, a geographic lon/lat CRS.
# VIIRS tiles are in EPSG:4326. For tile intersection and zonal extraction,
# the coordinate space is compatible and does not require a costly full reprojection.
if gdf_viirs.crs.is_geographic:
    viirs_crs_handling = (
        f"Used source geographic CRS ({gdf_viirs.crs}) without full reprojection; "
        "coordinates are longitude/latitude and compatible with VIIRS tile coordinates."
    )
else:
    # Fallback for unexpected projected inputs.
    # This branch is kept for robustness, but should not be used for the expected analysis-ready asset.
    gdf_viirs = gdf_viirs.to_crs(epsg=4326)
    viirs_crs_handling = "Reprojected minimal extraction GeoDataFrame to EPSG:4326."

viirs_prep_summary = pd.DataFrame(
    [
        {
            "tracts_available": len(gdf_viirs),
            "tract_crs_for_extraction": str(gdf_viirs.crs),
            "viirs_tile_count": len(viirs_tiles),
            "nightlight_field": nightlight_field,
            "crs_handling": viirs_crs_handling,
        }
    ]
)

display(viirs_prep_summary)

### 5.2 Extracting tract-level nighttime-light intensity

Instead of constructing a full national raster mosaic in memory, the workflow processes the VIIRS archive tile by tile, identifies census tracts intersecting each tile, calculates zonal means in manageable chunks, and joins the available values back to the tract layer. The number of raster files reflects only the local packaging of the archive and has no substantive interpretation.

**Runtime note.** This zonal-extraction stage is computationally intensive and may take several minutes, depending on the available memory, processor, and local storage performance.


In [ ]:
# ============================================================
# SECTION 5.2 — Tile-by-tile VIIRS extraction
# Memory-aware chunked version
# ============================================================

from shapely.geometry import box

if "gdf_viirs" not in globals():
    raise RuntimeError("GeoDataFrame 'gdf_viirs' is not available. Run the Section 5 setup cells first.")

if "viirs_tiles" not in globals():
    raise RuntimeError("VIIRS tiles are not available. Run Section 2 first.")

if len(viirs_tiles) == 0:
    raise RuntimeError("No VIIRS tiles were found in the staged archive.")

chunk_size = 1000

if nightlight_field in gdf_viirs.columns and gdf_viirs[nightlight_field].notna().any():
    extraction_mode = "reused_existing_field"
else:
    gdf_viirs = gdf_viirs.copy()
    gdf_viirs[nightlight_field] = np.nan
    extraction_mode = "computed_from_viirs_tiles_chunked"

    for tile_number, tile_path in enumerate(viirs_tiles, start=1):
        with rasterio.open(tile_path) as src:
            tile_bounds = src.bounds
            tile_bbox = box(
                tile_bounds.left,
                tile_bounds.bottom,
                tile_bounds.right,
                tile_bounds.top,
            )

            intersects_tile = gdf_viirs.geometry.intersects(tile_bbox)
            candidate_index = gdf_viirs.index[intersects_tile].to_numpy()

            if len(candidate_index) == 0:
                continue

            nodata_value = src.nodata if src.nodata is not None else -999

        print(
            f"Processing VIIRS tile {tile_number}/{len(viirs_tiles)}: "
            f"{tile_path.name} | candidate tracts: {len(candidate_index)}"
        )

        for start in range(0, len(candidate_index), chunk_size):
            chunk_index = candidate_index[start:start + chunk_size]
            gdf_chunk = gdf_viirs.loc[chunk_index, ["geometry"]].copy()

            stats = zonal_stats(
                vectors=gdf_chunk.geometry,
                raster=tile_path,
                stats=["mean"],
                nodata=nodata_value,
                geojson_out=False,
            )

            tile_means = pd.Series(
                [row["mean"] for row in stats],
                index=chunk_index,
                dtype="float64",
            )

            existing_values = gdf_viirs.loc[chunk_index, nightlight_field]
            gdf_viirs.loc[chunk_index, nightlight_field] = existing_values.combine_first(tile_means)

if nightlight_field in gdf.columns:
    gdf = gdf.drop(columns=[nightlight_field])

gdf = gdf.merge(
    gdf_viirs[["ct_id", nightlight_field]],
    on="ct_id",
    how="left",
)

nightlight_extraction_summary = pd.DataFrame(
    [
        {
            "extraction_mode": extraction_mode,
            "tiles_available": len(viirs_tiles),
            "chunk_size": chunk_size,
            "tracts_with_nonmissing_nightlight": int(gdf[nightlight_field].notna().sum()),
            "tracts_with_missing_nightlight": int(gdf[nightlight_field].isna().sum()),
            "share_nonmissing": round(float(gdf[nightlight_field].notna().mean()), 4),
        }
    ]
)

display(nightlight_extraction_summary)

The extraction produces tract-level nighttime-light means for observations with available VIIRS support. We inspect their distribution before comparing this contextual measure across the exploratory mobility-based access regimes.


In [ ]:
# ============================================================
# SECTION 5.2 — Distribution of extracted nightlight values
# ============================================================

nightlight_valid = gdf[nightlight_field].dropna()

nightlight_summary = pd.DataFrame(
    [
        {
            "count": int(nightlight_valid.shape[0]),
            "mean": round(float(nightlight_valid.mean()), 4),
            "median": round(float(nightlight_valid.median()), 4),
            "min": round(float(nightlight_valid.min()), 4),
            "max": round(float(nightlight_valid.max()), 4),
        }
    ]
)

display(nightlight_summary)

### 5.3 VIIRS context across mobility-based access regimes

We compared the distribution of tract-level VIIRS nighttime-light intensity across the four exploratory regimes, keeping this step descriptive because differences in radiance may reflect several local conditions that the mobility data do not observe.


In [ ]:
# ============================================================
# SECTION 5.3 — Regime-level economic profiles
# ============================================================

viirs_context_df = gdf.dropna(subset=["infrastructure_regime", nightlight_field]).copy()
viirs_context_df["infrastructure_regime"] = viirs_context_df["infrastructure_regime"].astype(int)

regime_nightlight_summary = (
    viirs_context_df.groupby("infrastructure_regime")[nightlight_field]
    .agg(["count", "mean", "median", "min", "max"])
    .round(4)
    .reset_index()
)

display(regime_nightlight_summary)

In [ ]:
# ============================================================
# SECTION 5.3 — Boxplot of nighttime light intensity by regime
# ============================================================

plt.figure(figsize=(8.5, 6))
sns.boxplot(
    data=viirs_context_df,
    x="infrastructure_regime",
    y=nightlight_field,
    palette="Greens",
)
plt.title("Nighttime Light Intensity by Mobility-Based Access Regime")
plt.xlabel("Mobility-based access regime")
plt.ylabel("Mean nighttime light intensity")
plt.tight_layout()
plt.show()

The regime summaries place the localized mobility configurations beside different distributions of nighttime-light intensity. This comparison describes external differentiation within the VIIRS-covered tracts, but it does not turn the regimes into economic-development or market-performance classes.


### 5.4 Held-out comparison with a visit-volume baseline

Visit volume served as the reference for examining whether the multidimensional mobility representation contained additional descriptive information about VIIRS intensity. From tracts with complete standardized log visit volume, normalized MII, regime assignment, and nighttime-light intensity, we draw a reproducible random sample of 120,000 observations with seed 42 and retain the same 84,000 training and 36,000 test observations across four linear specifications.

The specifications use visit volume alone, MII alone, both measures together, and both measures with regime indicators. Numerical predictors are standardized within each pipeline, while held-out `R²`, RMSE, and MAE provide a common diagnostic comparison. The random split evaluates descriptive fit within the sampled VIIRS-covered tracts and does not support causal inference, population-level estimation, construct validation, or prediction in geographically independent areas.


In [ ]:
# ============================================================
# SECTION 5.4 — Held-out comparison with a visit-volume baseline
# ============================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression

required_comparison_fields = [
    nightlight_field,
    mobility_intensity_var,
    "mii_norm",
    "infrastructure_regime",
]

missing_comparison_fields = [
    col for col in required_comparison_fields
    if col not in gdf.columns
]

if missing_comparison_fields:
    raise RuntimeError(
        "The held-out comparison fields are missing: "
        + ", ".join(missing_comparison_fields)
    )

comparison_df = gdf.dropna(subset=required_comparison_fields).copy()
comparison_df["infrastructure_regime"] = comparison_df["infrastructure_regime"].astype(int).astype(str)

comparison_sample_size = 120_000
if len(comparison_df) < comparison_sample_size:
    raise RuntimeError(
        f"The held-out comparison requires {comparison_sample_size:,} complete tracts; "
        f"only {len(comparison_df):,} are available."
    )

comparison_sample = comparison_df.sample(
    n=comparison_sample_size,
    random_state=response_random_state,
).copy()

train_df, test_df = train_test_split(
    comparison_sample,
    test_size=0.30,
    random_state=response_random_state,
)

model_specs = [
    {
        "model": "Visit-volume intensity only",
        "numeric_features": [mobility_intensity_var],
        "categorical_features": [],
    },
    {
        "model": "MII only",
        "numeric_features": ["mii_norm"],
        "categorical_features": [],
    },
    {
        "model": "Visit-volume intensity + MII",
        "numeric_features": [mobility_intensity_var, "mii_norm"],
        "categorical_features": [],
    },
    {
        "model": "Visit-volume intensity + MII + regimes",
        "numeric_features": [mobility_intensity_var, "mii_norm"],
        "categorical_features": ["infrastructure_regime"],
    },
]

heldout_results = []

for spec in model_specs:
    numeric_features = spec["numeric_features"]
    categorical_features = spec["categorical_features"]

    transformers = []

    if numeric_features:
        transformers.append(
            (
                "numeric",
                StandardScaler(),
                numeric_features,
            )
        )

    if categorical_features:
        transformers.append(
            (
                "categorical",
                OneHotEncoder(drop="first", handle_unknown="ignore"),
                categorical_features,
            )
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )

    model = make_pipeline(
        preprocessor,
        LinearRegression(),
    )

    X_train = train_df[numeric_features + categorical_features]
    y_train = train_df[nightlight_field].to_numpy(dtype="float64")

    X_test = test_df[numeric_features + categorical_features]
    y_test = test_df[nightlight_field].to_numpy(dtype="float64")

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    heldout_results.append(
        {
            "model": spec["model"],
            "train_observations": len(train_df),
            "test_observations": len(test_df),
            "r2_test": round(float(r2), 4),
            "rmse_test": round(float(rmse), 4),
            "mae_test": round(float(mae), 4),
            "numeric_features": ", ".join(numeric_features) if numeric_features else "None",
            "categorical_features": ", ".join(categorical_features) if categorical_features else "None",
        }
    )

heldout_comparison_summary = pd.DataFrame(heldout_results)

visit_volume_only_r2 = heldout_comparison_summary.loc[
    heldout_comparison_summary["model"] == "Visit-volume intensity only",
    "r2_test",
].iloc[0]

heldout_comparison_summary["r2_gain_vs_visit_volume_only"] = (
    heldout_comparison_summary["r2_test"] - visit_volume_only_r2
).round(4)

display(heldout_comparison_summary)

plt.figure(figsize=(8.5, 5.2))
sns.barplot(
    data=heldout_comparison_summary,
    x="r2_test",
    y="model",
    color="#4C78A8",
)
plt.axvline(
    visit_volume_only_r2,
    color="#8B1A1A",
    linestyle="--",
    linewidth=1.6,
    label="Visit-volume-only baseline",
)
plt.xlabel("Held-out test R²")
plt.ylabel("")
plt.title("Held-Out VIIRS Comparison Against a Visit-Volume Baseline")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

Among the simple specifications, visit volume provided the strongest standalone description of VIIRS intensity, with held-out `R² = 0.3799`, compared with `0.2529` for the MII-only specification. Combining visit volume with the MII changed held-out `R²` to `0.3904`, while adding the regime indicators changed it to `0.3919`; RMSE and MAE declined slightly across the same sequence.

The multidimensional representation consequently added modest descriptive information to the visit-volume reference without replacing it. Because these models use a random split of VIIRS-covered tracts, the comparison remains a diagnostic of the observed sample rather than evidence that the MII predicts market outcomes or transfers to new geographic areas.


### 5.5 Regime-specific associations between visit volume and VIIRS intensity

We fitted a separate cubic-spline profile within each exploratory regime to examine whether standardized log visit volume had the same descriptive association with nighttime-light intensity across local mobility configurations. Each profile uses five knots and a third-degree basis, and its interpretation is restricted to the mobility range represented by observations in that regime.


In [ ]:
# ============================================================
# SECTION 5.5 — Helper function for spline-based response estimation
# ============================================================

def fit_regime_spline(x: np.ndarray, y: np.ndarray, n_knots: int = 5, degree: int = 3):
    x = np.asarray(x).reshape(-1, 1)
    y = np.asarray(y)

    spline = SplineTransformer(
        degree=degree,
        n_knots=n_knots,
        include_bias=False,
    )
    X_spline = spline.fit_transform(x)

    model = LinearRegression()
    model.fit(X_spline, y)

    x_grid = np.linspace(x.min(), x.max(), 200).reshape(-1, 1)
    y_pred = model.predict(spline.transform(x_grid))

    r2 = model.score(X_spline, y)

    return {
        "model": model,
        "spline": spline,
        "x_grid": x_grid.flatten(),
        "y_pred": y_pred,
        "r2": float(r2),
        "n_obs": int(len(y)),
        "x_min": float(x.min()),
        "x_max": float(x.max()),
    }

In [ ]:
# ============================================================
# SECTION 5.5 — Estimate regime-specific response curves
# ============================================================

response_df = gdf.dropna(
    subset=["infrastructure_regime", mobility_intensity_var, nightlight_field]
).copy()
response_df["infrastructure_regime"] = response_df["infrastructure_regime"].astype(int)

regime_models = {}
regime_diagnostics = []

for regime_id in sorted(response_df["infrastructure_regime"].unique()):
    regime_subset = response_df.loc[response_df["infrastructure_regime"] == regime_id]

    model_result = fit_regime_spline(
        x=regime_subset[mobility_intensity_var].values,
        y=regime_subset[nightlight_field].values,
        n_knots=5,
        degree=3,
    )

    regime_models[regime_id] = model_result

    regime_diagnostics.append(
        {
            "regime": regime_id,
            "n_obs": model_result["n_obs"],
            "x_min": round(model_result["x_min"], 3),
            "x_max": round(model_result["x_max"], 3),
            "r2": round(model_result["r2"], 4),
        }
    )

regime_diagnostics_df = pd.DataFrame(regime_diagnostics)
display(regime_diagnostics_df)

In [ ]:
# ============================================================
# SECTION 5.5 — Plot regime-specific visit-volume and VIIRS profiles
# ============================================================

plt.figure(figsize=(10, 6.5))

for regime_id, model_result in regime_models.items():
    plt.plot(
        model_result["x_grid"],
        model_result["y_pred"],
        linewidth=2.5,
        label=f"Regime {regime_id}",
    )

plt.title("Fitted Associations Between Visit Volume and VIIRS Nighttime Light Intensity")
plt.xlabel("Standardized log visit volume")
plt.ylabel("Fitted VIIRS nighttime light intensity")
plt.legend(title="Mobility-based access regime")
plt.tight_layout()
plt.show()

The fitted curves differed across regimes within their observed mobility ranges, showing that a common visit-volume measure appeared beside different nighttime-light contexts across the exploratory configurations. These profiles remain descriptive, depend on the support available within each regime, and do not show that a regime caused greater economic intensity.

The next section changes the spatial support of the MII within the São Paulo metropolitan window, using VIIRS as an ancillary allocation surface rather than as independent external context.


# 6. Market Diagnosis Across Spatial Supports

Census-tract values represent every location inside a polygon through the same aggregate score, even when the ancillary spatial context varies internally. We used the São Paulo metropolitan window to examine what one explicit change in spatial support produces, allocating tract-level MII values to a 100 m grid through normalized within-tract VIIRS weights while preserving the original aggregate value of every represented tract.

The resulting grid is a conditional, model-based allocation. It does not contain mobility observed at 100 m, create new aggregate information, or independently validate the MII with the same VIIRS surface used to determine its internal weights.


In [ ]:
# ============================================================
# SECTION 6 — Imports and configuration
# ============================================================

from contextlib import ExitStack

from shapely.geometry import box
from rasterio.features import rasterize
from rasterio.merge import merge
from rasterio.transform import from_origin
from rasterio.warp import reproject, Resampling
from matplotlib.colors import TwoSlopeNorm
from matplotlib.cm import ScalarMappable

sp_bbox = (-47.25, -24.20, -45.70, -23.10)
grid_resolution_m = 100
focus_buffer_m = 4000
allocation_plot_crs = "EPSG:5880"

### 6.1 Selecting the metropolitan study window

The allocation uses a São Paulo metropolitan window selected from the national tract layer before reprojection to a metric coordinate system. This order limits memory use while retaining the tract identifiers, MII fields, and geometries required for construction of the 100 m grid.


In [ ]:
# ============================================================
# SECTION 6.1 — Select São Paulo metropolitan window
# Memory-aware version: subset before reprojection
# ============================================================

if "gdf" not in globals():
    raise RuntimeError("GeoDataFrame 'gdf' is not available. Run the earlier sections first.")

if gdf.crs is None:
    raise RuntimeError("The tract GeoDataFrame has no CRS. The São Paulo selection requires geographic coordinates.")

required_sp_columns = ["ct_id", "mii", "mii_norm", "geometry"]
missing_sp_columns = [col for col in required_sp_columns if col not in gdf.columns]

if missing_sp_columns:
    raise ValueError(
        "The GeoPackage is missing required São Paulo allocation fields: "
        + ", ".join(missing_sp_columns)
    )

# The shared GeoPackage is already in a geographic lon/lat CRS.
# Avoid reprojecting the full national GeoDataFrame. Select the São Paulo
# window first, then reproject only that subset.
if not gdf.crs.is_geographic:
    raise RuntimeError(
        "Expected a geographic CRS for bbox selection. "
        f"Found {gdf.crs}. Reprojection of the full national file is intentionally avoided."
    )

lon_min, lat_min, lon_max, lat_max = sp_bbox

gdf_geo = gdf.loc[:, required_sp_columns].copy()
sp_geo = gdf_geo.cx[lon_min:lon_max, lat_min:lat_max].copy()

if sp_geo.empty:
    raise RuntimeError("The São Paulo study window returned no census tracts.")

sp_metric = sp_geo.to_crs(allocation_plot_crs).copy()
sp_metric["tract_code"] = np.arange(1, len(sp_metric) + 1)

sp_summary = pd.DataFrame(
    [
        {
            "study_area": "São Paulo metropolitan window",
            "bbox_wgs84": str(sp_bbox),
            "selected_tracts": len(sp_geo),
            "geographic_crs": str(sp_geo.crs),
            "metric_crs": allocation_plot_crs,
            "grid_resolution_m": grid_resolution_m,
            "platform_safety_note": "Subset was selected before reprojection to avoid full national geometry transformation.",
        }
    ]
)

display(sp_summary)

### 6.2 Preparing the ancillary VIIRS surface at 100 m

Only VIIRS raster files intersecting the selected window are merged and reprojected to the 100 m grid. Their cell values provide the within-tract weights used in the allocation and are not interpreted as fine-scale observations of mobility-based market infrastructure.


In [ ]:
# ============================================================
# SECTION 6.2 — Select VIIRS tiles intersecting the study area
# ============================================================

study_bbox_geom = box(*sp_geo.total_bounds)

intersecting_viirs_tiles = []

for tile_path in viirs_tiles:
    with rasterio.open(tile_path) as src:
        tile_bounds = src.bounds
        tile_geom = box(tile_bounds.left, tile_bounds.bottom, tile_bounds.right, tile_bounds.top)
        if tile_geom.intersects(study_bbox_geom):
            intersecting_viirs_tiles.append(tile_path)

if len(intersecting_viirs_tiles) == 0:
    raise RuntimeError("No VIIRS tiles intersected the Sao Paulo study window.")

viirs_window_summary = pd.DataFrame(
    [
        {
            "intersecting_viirs_tiles": len(intersecting_viirs_tiles),
            "first_tile": intersecting_viirs_tiles[0].name,
        }
    ]
)

display(viirs_window_summary)

In [ ]:
# ============================================================
# SECTION 6.2 — Build 100 m grid and reproject VIIRS
# ============================================================

xmin, ymin, xmax, ymax = sp_metric.total_bounds
width = int(np.ceil((xmax - xmin) / grid_resolution_m))
height = int(np.ceil((ymax - ymin) / grid_resolution_m))
target_transform = from_origin(xmin, ymax, grid_resolution_m, grid_resolution_m)

with ExitStack() as stack:
    src_datasets = [stack.enter_context(rasterio.open(tile)) for tile in intersecting_viirs_tiles]
    merged_viirs, merged_transform = merge(src_datasets)

merged_viirs_band = merged_viirs[0]

viirs_100m = np.full((height, width), np.nan, dtype="float32")

reproject(
    source=merged_viirs_band,
    destination=viirs_100m,
    src_transform=merged_transform,
    src_crs="EPSG:4326",
    dst_transform=target_transform,
    dst_crs=allocation_plot_crs,
    dst_nodata=np.nan,
    resampling=Resampling.bilinear,
)

grid_summary = pd.DataFrame(
    [
        {
            "grid_width_cells": width,
            "grid_height_cells": height,
            "grid_total_cells": int(width * height),
            "grid_resolution_m": grid_resolution_m,
            "viirs_nonmissing_cells": int(np.isfinite(viirs_100m).sum()),
        }
    ]
)

display(grid_summary)

### 6.3 Aggregate-preserving tract-to-grid allocation

We then assigned each 100-meter grid cell to a census tract and redistributed tract-level MII values across the grid using normalized VIIRS-derived weights.

For rasterized tracts on the target grid, the within-tract weights were normalized so that they sum to 1. This preserved each represented tract's aggregate MII value after allocation to the grid. When a tract contained no positive ancillary signal, we fell back to a uniform allocation rule within that tract.

In [ ]:
# ============================================================
# SECTION 6.3 — Rasterize tracts and compute aggregate-preserving allocation
# ============================================================

tract_raster = rasterize(
    shapes=zip(sp_metric.geometry, sp_metric["tract_code"]),
    out_shape=(height, width),
    transform=target_transform,
    fill=0,
    dtype="int32",
)

valid_cell_mask = tract_raster > 0
tract_codes_flat = tract_raster[valid_cell_mask].astype(np.int32)

viirs_weights = np.where(
    np.isfinite(viirs_100m) & (viirs_100m > 0),
    viirs_100m,
    0.0,
)

viirs_weights_flat = viirs_weights[valid_cell_mask]

max_code = int(sp_metric["tract_code"].max())

pixel_counts = np.bincount(
    tract_codes_flat,
    minlength=max_code + 1,
)

weight_sums = np.bincount(
    tract_codes_flat,
    weights=viirs_weights_flat,
    minlength=max_code + 1,
)

mii_by_code = np.zeros(max_code + 1, dtype="float64")
mii_norm_by_code = np.zeros(max_code + 1, dtype="float64")

mii_by_code[sp_metric["tract_code"].values] = sp_metric["mii"].values
mii_norm_by_code[sp_metric["tract_code"].values] = sp_metric["mii_norm"].values

sum_lookup = weight_sums[tract_codes_flat]
count_lookup = pixel_counts[tract_codes_flat]

normalized_weights_flat = np.empty_like(viirs_weights_flat, dtype="float64")

positive_weight_mask = sum_lookup > 0
normalized_weights_flat[positive_weight_mask] = (
    viirs_weights_flat[positive_weight_mask] / sum_lookup[positive_weight_mask]
)

uniform_weight_mask = ~positive_weight_mask
normalized_weights_flat[uniform_weight_mask] = (
    1.0 / np.maximum(count_lookup[uniform_weight_mask], 1)
)

mii_100m_flat = mii_by_code[tract_codes_flat] * normalized_weights_flat
mii_norm_100m_flat = mii_norm_by_code[tract_codes_flat] * normalized_weights_flat

mii_100m = np.full((height, width), np.nan, dtype="float64")
mii_norm_100m = np.full((height, width), np.nan, dtype="float64")

mii_100m[valid_cell_mask] = mii_100m_flat
mii_norm_100m[valid_cell_mask] = mii_norm_100m_flat

# ------------------------------------------------------------
# Aggregate-preservation check
# ------------------------------------------------------------
# The algebraic preservation check should be evaluated only for
# tracts that are actually represented by at least one rasterized
# 100 m grid cell. Very small edge/intersection geometries can be
# selected by the bbox but not represented on the target grid.

allocated_sum_by_tract = np.bincount(
    tract_codes_flat,
    weights=mii_100m_flat,
    minlength=max_code + 1,
)

original_sum_by_tract = np.zeros(max_code + 1, dtype="float64")
original_sum_by_tract[sp_metric["tract_code"].values] = sp_metric["mii"].values

represented_tract_codes = sp_metric.loc[
    pixel_counts[sp_metric["tract_code"].values] > 0,
    "tract_code",
].values

preservation_check_df = pd.DataFrame(
    {
        "tract_code": represented_tract_codes,
        "original_mii": original_sum_by_tract[represented_tract_codes],
        "allocated_mii_sum": allocated_sum_by_tract[represented_tract_codes],
        "rasterized_cell_count": pixel_counts[represented_tract_codes],
    }
)

preservation_check_df["absolute_difference"] = (
    preservation_check_df["allocated_mii_sum"] - preservation_check_df["original_mii"]
).abs()

allocation_preservation_summary = pd.DataFrame(
    [
        {
            "selected_tracts": int(len(sp_metric)),
            "tracts_represented_on_grid": int(len(preservation_check_df)),
            "tracts_not_represented_on_grid": int(len(sp_metric) - len(preservation_check_df)),
            "total_original_mii_for_represented_tracts": round(float(preservation_check_df["original_mii"].sum()), 6),
            "total_allocated_mii_for_represented_tracts": round(float(preservation_check_df["allocated_mii_sum"].sum()), 6),
            "max_absolute_tract_difference": round(float(preservation_check_df["absolute_difference"].max()), 10),
            "mean_absolute_tract_difference": round(float(preservation_check_df["absolute_difference"].mean()), 10),
        }
    ]
)

display(allocation_preservation_summary)
display(preservation_check_df.head())

The algebraic check is evaluated only for tracts represented by at least one rasterized cell. Within that set, the allocated cell values sum to the original tract-level MII up to numerical precision, while selected tracts that receive no cell at the chosen resolution remain outside the preservation calculation.

This equality verifies the allocation rule rather than the true internal distribution of access, because the MII is dimensionless and VIIRS determines how its tract-level value is divided among cells.


### 6.4 Comparing tract-level values with the allocated surface

The following views place the original tract-level MII beside its 100 m allocation. Each tract retains the same aggregate value when its represented cells are added together, although VIIRS weighting produces unequal modeled concentrations inside the polygon.


In [ ]:
# ============================================================
# SECTION 6.4 — Manual-bbox comparison: tract-level vs 100 m surface
# ============================================================

from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

# ------------------------------------------------------------
# 1. Manual focus window (EDIT HERE IF NEEDED)
# Coordinates in the metric CRS used by sp_metric
# ------------------------------------------------------------
# Initial suggestion: use the current urban core area and fine-tune if needed
fxmin, fymin, fxmax, fymax = (
    xmin + 0.18 * (xmax - xmin),
    ymin + 0.22 * (ymax - ymin),
    xmin + 0.52 * (xmax - xmin),
    ymin + 0.62 * (ymax - ymin),
)

focus_window = box(fxmin, fymin, fxmax, fymax)
sp_focus = sp_metric.loc[sp_metric.geometry.intersects(focus_window)].copy()

if sp_focus.empty:
    raise RuntimeError("The manual bounding box returned no tracts. Adjust fxmin, fymin, fxmax, fymax.")

focus_outline = sp_focus.dissolve()

# ------------------------------------------------------------
# 2. Raster crop for the same bbox
# ------------------------------------------------------------
row_min = max(0, int((ymax - fymax) / grid_resolution_m))
row_max = min(height, int(np.ceil((ymax - fymin) / grid_resolution_m)))
col_min = max(0, int((fxmin - xmin) / grid_resolution_m))
col_max = min(width, int(np.ceil((fxmax - xmin) / grid_resolution_m)))

mii_100m_focus = mii_norm_100m[row_min:row_max, col_min:col_max]

raster_extent = (
    xmin + col_min * grid_resolution_m,
    xmin + col_max * grid_resolution_m,
    ymax - row_max * grid_resolution_m,
    ymax - row_min * grid_resolution_m,
)

raster_values = mii_100m_focus[np.isfinite(mii_100m_focus)]
if raster_values.size == 0:
    raise RuntimeError("The selected bbox produced no valid allocated raster values. Adjust the bbox.")

# ------------------------------------------------------------
# 3. Color scales
# ------------------------------------------------------------
# Left panel: fixed normalized scale for conceptual consistency
left_vmin, left_vmax = -1.0, 1.0

# Right panel: local stretch to make 100 m variation visible
right_vmin = float(np.nanpercentile(raster_values, 2))
right_vmax = float(np.nanpercentile(raster_values, 98))

if right_vmin == right_vmax:
    right_vmin = float(np.nanmin(raster_values))
    right_vmax = float(np.nanmax(raster_values))

# ------------------------------------------------------------
# 4. Figure
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 8), facecolor="white")

cmap_name = "viridis"

# Left panel — tract polygons
sp_focus.plot(
    column="mii_norm",
    cmap=cmap_name,
    linewidth=0.7,
    edgecolor="#2f2f2f",
    ax=axes[0],
    vmin=left_vmin,
    vmax=left_vmax,
)

focus_outline.boundary.plot(
    ax=axes[0],
    color="black",
    linewidth=1.1,
)

axes[0].set_xlim(fxmin, fxmax)
axes[0].set_ylim(fymin, fymax)
axes[0].set_title("Tract-Level Infrastructure (Normalized MII)", fontsize=14, pad=12)
axes[0].set_axis_off()

sm_left = ScalarMappable(
    norm=Normalize(vmin=left_vmin, vmax=left_vmax),
    cmap=cmap_name,
)
sm_left._A = []

cbar_left = fig.colorbar(
    sm_left,
    ax=axes[0],
    fraction=0.046,
    pad=0.02,
)
cbar_left.set_label("Normalized MII", fontsize=11)

# Right panel — 100 m raster
axes[1].imshow(
    mii_100m_focus,
    cmap=cmap_name,
    extent=raster_extent,
    origin="upper",
    interpolation="nearest",
    vmin=right_vmin,
    vmax=right_vmax,
)

focus_outline.boundary.plot(
    ax=axes[1],
    color="black",
    linewidth=1.1,
)

axes[1].set_xlim(fxmin, fxmax)
axes[1].set_ylim(fymin, fymax)
axes[1].set_title("100 m Aggregate-Preserving Allocation", fontsize=14, pad=12)
axes[1].set_axis_off()

sm_right = ScalarMappable(
    norm=Normalize(vmin=right_vmin, vmax=right_vmax),
    cmap=cmap_name,
)
sm_right._A = []

cbar_right = fig.colorbar(
    sm_right,
    ax=axes[1],
    fraction=0.046,
    pad=0.02,
)
cbar_right.set_label("Allocated 100 m Surface", fontsize=11)

fig.suptitle(
    "High-Zoom Comparison Between Tract-Level Structure and 100 m Allocation",
    fontsize=18,
    fontweight="bold",
    y=0.98,
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

# ------------------------------------------------------------
# 5. Optional diagnostic table
# ------------------------------------------------------------
focus_summary = pd.DataFrame(
    [
        {
            "selected_tracts": len(sp_focus),
            "left_scale": "fixed [-1, 1]",
            "right_scale_min": round(right_vmin, 4),
            "right_scale_max": round(right_vmax, 4),
        }
    ]
)

display(focus_summary)

In [ ]:
# ============================================================
# SECTION 6.4 — Manual zoom-in on the most informative local area
# Reduced zoom version
# ============================================================

from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

# ------------------------------------------------------------
# 1. Manual zoom-in bbox
# Adjust these values if needed
# ------------------------------------------------------------
focus_width = 5200
focus_height = 4400

# Same center as before; larger window for less aggressive zoom
cx = xmin + 0.52 * (xmax - xmin)
cy = ymin + 0.60 * (ymax - ymin)

fxmin = cx - focus_width / 2
fxmax = cx + focus_width / 2
fymin = cy - focus_height / 2
fymax = cy + focus_height / 2

focus_window = box(fxmin, fymin, fxmax, fymax)
sp_focus = sp_metric.loc[sp_metric.geometry.intersects(focus_window)].copy()

if sp_focus.empty:
    raise RuntimeError("The zoom-in bbox returned no tracts. Adjust cx, cy, focus_width, or focus_height.")

focus_outline = sp_focus.dissolve()

# ------------------------------------------------------------
# 2. Raster crop for the same bbox
# ------------------------------------------------------------
row_min = max(0, int((ymax - fymax) / grid_resolution_m))
row_max = min(height, int(np.ceil((ymax - fymin) / grid_resolution_m)))
col_min = max(0, int((fxmin - xmin) / grid_resolution_m))
col_max = min(width, int(np.ceil((fxmax - xmin) / grid_resolution_m)))

mii_100m_focus = mii_norm_100m[row_min:row_max, col_min:col_max]

raster_extent = (
    xmin + col_min * grid_resolution_m,
    xmin + col_max * grid_resolution_m,
    ymax - row_max * grid_resolution_m,
    ymax - row_min * grid_resolution_m,
)

raster_values = mii_100m_focus[np.isfinite(mii_100m_focus)]
if raster_values.size == 0:
    raise RuntimeError("The selected zoom-in bbox produced no valid raster values.")

# ------------------------------------------------------------
# 3. Color scales
# Left: fixed normalized scale
# Right: local stretch to reveal 100 m variation
# ------------------------------------------------------------
left_vmin, left_vmax = -1.0, 1.0

right_vmin = float(np.nanpercentile(raster_values, 2))
right_vmax = float(np.nanpercentile(raster_values, 98))

if right_vmin == right_vmax:
    right_vmin = float(np.nanmin(raster_values))
    right_vmax = float(np.nanmax(raster_values))

# ------------------------------------------------------------
# 4. Figure
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 8), facecolor="white")

cmap_name = "viridis"

# Left panel — tract polygons
sp_focus.plot(
    column="mii_norm",
    cmap=cmap_name,
    linewidth=0.75,
    edgecolor="#2f2f2f",
    ax=axes[0],
    vmin=left_vmin,
    vmax=left_vmax,
)

focus_outline.boundary.plot(
    ax=axes[0],
    color="black",
    linewidth=1.1,
)

axes[0].set_xlim(fxmin, fxmax)
axes[0].set_ylim(fymin, fymax)
axes[0].set_title("Tract-Level Infrastructure (Normalized MII)", fontsize=14, pad=12)
axes[0].set_axis_off()

sm_left = ScalarMappable(
    norm=Normalize(vmin=left_vmin, vmax=left_vmax),
    cmap=cmap_name,
)
sm_left._A = []

cbar_left = fig.colorbar(
    sm_left,
    ax=axes[0],
    fraction=0.046,
    pad=0.02,
)
cbar_left.set_label("Normalized MII", fontsize=11)

# Right panel — 100 m raster
axes[1].imshow(
    mii_100m_focus,
    cmap=cmap_name,
    extent=raster_extent,
    origin="upper",
    interpolation="nearest",
    vmin=right_vmin,
    vmax=right_vmax,
)

focus_outline.boundary.plot(
    ax=axes[1],
    color="black",
    linewidth=1.1,
)

axes[1].set_xlim(fxmin, fxmax)
axes[1].set_ylim(fymin, fymax)
axes[1].set_title("100 m Aggregate-Preserving Allocation", fontsize=14, pad=12)
axes[1].set_axis_off()

sm_right = ScalarMappable(
    norm=Normalize(vmin=right_vmin, vmax=right_vmax),
    cmap=cmap_name,
)
sm_right._A = []

cbar_right = fig.colorbar(
    sm_right,
    ax=axes[1],
    fraction=0.046,
    pad=0.02,
)
cbar_right.set_label("Allocated 100 m Surface", fontsize=11)

fig.suptitle(
    "High-Zoom Comparison Between Tract-Level Structure and 100 m Allocation",
    fontsize=18,
    fontweight="bold",
    y=0.98,
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

# ------------------------------------------------------------
# 5. Diagnostic table
# ------------------------------------------------------------
focus_summary = pd.DataFrame(
    [
        {
            "selected_tracts": len(sp_focus),
            "bbox_center_x": round(cx, 2),
            "bbox_center_y": round(cy, 2),
            "bbox_width_m": focus_width,
            "bbox_height_m": focus_height,
            "left_scale": "fixed [-1, 1]",
            "right_scale_min": round(right_vmin, 4),
            "right_scale_max": round(right_vmax, 4),
        }
    ]
)

display(focus_summary)

The two views illustrate the same change in spatial support at different scales. The tract layer repeats one MII value throughout each polygon, whereas the allocated grid distributes that value unevenly under the selected VIIRS weighting rule, without changing the aggregate value of represented tracts.

The visible within-tract pattern is conditional on the grid, rasterization, and ancillary surface, so it should not be read as observed sub-tract mobility or as a ranking of fine-scale market opportunity.


### 6.5 Within-tract allocation-dispersion diagnostic

We summarize how unevenly each represented tract's allocated cell values are distributed by calculating their within-tract standard deviation. The statistic describes dispersion produced by the allocation rule and is sensitive to the number of cells representing each tract; it is not an observed measure of internal market fragmentation.


In [ ]:
# ============================================================
# SECTION 6.5 — Allocation-dispersion diagnostics by tract
# ============================================================

valid_relative_mask = valid_cell_mask & np.isfinite(mii_norm_100m)

tract_codes_dispersion = tract_raster[valid_relative_mask].astype(np.int32)
relative_intensity_flat = mii_norm_100m[valid_relative_mask]

allocation_dispersion_df = pd.DataFrame(
    {
        "tract_code": tract_codes_dispersion,
        "relative_intensity": relative_intensity_flat,
    }
)

tract_allocation_dispersion = (
    allocation_dispersion_df.groupby("tract_code")["relative_intensity"]
    .agg(["count", "std", "min", "max"])
    .reset_index()
    .rename(columns={"std": "allocation_dispersion"})
)

sp_metric = sp_metric.drop(
    columns=[col for col in ["count", "allocation_dispersion", "min", "max"] if col in sp_metric.columns],
    errors="ignore",
)

sp_metric = sp_metric.merge(
    tract_allocation_dispersion[["tract_code", "count", "allocation_dispersion", "min", "max"]],
    on="tract_code",
    how="left",
)

allocation_dispersion_summary = (
    sp_metric[["tract_code", "ct_id", "allocation_dispersion", "count", "mii", "mii_norm"]]
    .sort_values("allocation_dispersion", ascending=False)
    .head(10)
    .round(4)
)

display(allocation_dispersion_summary)

In [ ]:
# ============================================================
# SECTION 6.5 — Local view of the tract with highest allocation dispersion
# ============================================================

if "allocation_dispersion" not in sp_metric.columns:
    raise RuntimeError("Allocation-dispersion diagnostics are not available. Run the previous cell first.")

highest_dispersion_tract = sp_metric.dropna(subset=["allocation_dispersion"]).sort_values(
    "allocation_dispersion", ascending=False
).iloc[0]

focus_geom = highest_dispersion_tract.geometry.buffer(focus_buffer_m)
fxmin, fymin, fxmax, fymax = focus_geom.bounds

fig, ax = plt.subplots(figsize=(8, 8), facecolor="white")

ax.imshow(
    mii_norm_100m,
    cmap="viridis",
    extent=(xmin, xmax, ymin, ymax),
    origin="upper",
    interpolation="nearest",
)

sp_metric.boundary.plot(ax=ax, color="white", linewidth=0.12)
sp_metric.loc[[highest_dispersion_tract.name]].boundary.plot(ax=ax, color="black", linewidth=1.3)

ax.set_xlim(fxmin, fxmax)
ax.set_ylim(fymin, fymax)
ax.set_title("Local View of the Highest Allocation Dispersion", fontsize=14, pad=12)
ax.set_axis_off()

plt.tight_layout()
plt.show()

The dispersion output identifies tracts for which the selected grid and VIIRS weighting rule produce more uneven cell allocations. Tracts represented by very few cells can appear extreme, so this diagnostic is used to illustrate sensitivity to spatial support rather than to rank observed sub-tract heterogeneity.


# 7. Scope, Reproducibility, and Analytical Summary

The workflow combines internal covariance checks, spatial diagnostics, localized sensitivity analyses, bounded comparison with VIIRS, and a transparent change in spatial support. These stages assess what aggregated mobility can reveal about pre-outcome access conditions without converting the MII into demand, the regimes into market types, VIIRS into performance, or allocated grid cells into observed 100 m mobility.

## Diagnostic and Robustness Summary

| Analytical layer | Evidence provided | Interpretation boundary |
|---|---|---|
| **Global covariance structure** | PC1 explained `61.43%` of variation across the seven standardized indicators. | A dominant shared dimension does not make all indicators interchangeable. |
| **MII alignment** | Recomputed PC1 correlated `0.9447` with the retained MII. | Shared inputs make this an internal covariance-alignment check, not complete construct validation. |
| **Global spatial organization** | KNN Moran's I for normalized MII was `0.4550` with `p_sim = .01`. | The statistic describes global dependence under the eight-nearest-neighbor definition and does not identify local clusters or spillovers. |
| **Localized robustness** | Local PC1 scores remained closely aligned under distance weighting and neighborhoods of 100 and 300 tracts. | Finer local dimensions were more sensitive to the neighborhood specification. |
| **Exploratory access regimes** | Four recurring configurations organized focal PC1 and PC2 scores for mapping and comparison. | The regimes are conditional classifications, not natural market types or rankings. |
| **VIIRS context** | Regimes differed in nighttime-light distributions and fitted visit-volume profiles. | VIIRS is contextual proxy evidence, not sales, demand, conversion, or performance. |
| **Held-out comparison** | Adding MII and regime indicators produced small gains relative to visit volume alone. | The random split describes the sampled VIIRS-covered tracts and does not establish geographic transferability. |
| **Aggregate-preserving allocation** | Represented tract totals were preserved after allocation to the 100 m grid. | The grid values are model-based shares of a dimensionless tract index, not observed fine-resolution mobility. |


In [ ]:
# ============================================================
# SECTION 7 — Executive summary table
# ============================================================

summary_rows = []

if "explained_variance_df" in globals():
    summary_rows.append(
        {
            "stage": "Dominant covariance structure",
            "result": f"PC1 explained {explained_variance_df.loc[0, 'explained_variance_ratio']:.4f} of total variance",
        }
    )

if "alignment_summary" in globals():
    summary_rows.append(
        {
            "stage": "MII consistency check",
            "result": f"Recomputed PC1 and stored MII correlation = {alignment_summary.loc[0, 'correlation_between_recomputed_pc1_and_stored_mii']:.4f}",
        }
    )

if "regime_assignment_summary" in globals():
    n_regimes_found = int(regime_assignment_summary["infrastructure_regime"].nunique())
    summary_rows.append(
        {
            "stage": "Exploratory access regimes",
            "result": f"{n_regimes_found} tract-level regimes were identified and propagated across valid census tracts",
        }
    )
elif "tract_regime_counts" in globals():
    n_regimes_found = int(tract_regime_counts["infrastructure_regime"].dropna().nunique())
    summary_rows.append(
        {
            "stage": "Exploratory access regimes",
            "result": f"{n_regimes_found} tract-level regimes were identified and propagated across valid census tracts",
        }
    )

if "regime_nightlight_summary" in globals():
    summary_rows.append(
        {
            "stage": "External comparison",
            "result": "Nighttime light intensity differed across mobility-based access regimes",
        }
    )

if "allocation_preservation_summary" in globals():
    summary_rows.append(
        {
            "stage": "Multiscale allocation",
            "result": "Sum-preserving allocation was completed for rasterized tracts represented on the target grid",
        }
    )

if "allocation_dispersion_summary" in globals():
    summary_rows.append(
        {
            "stage": "Within-tract allocation dispersion",
            "result": "The selected grid and VIIRS weights produced conditional within-tract dispersion",
        }
    )

executive_summary_df = pd.DataFrame(summary_rows)
display(executive_summary_df)

### 7.1 Key takeaways

The notebook supports a bounded diagnostic interpretation of aggregated mobility. The seven indicators contained a dominant covariance dimension that was closely aligned with the retained equal-weight MII, while localized analyses showed that this organization was not identical across places and that the leading local dimension remained comparatively stable under alternative weighting and neighborhood choices.

The four access regimes organize recurring local configurations without ranking them, and their VIIRS comparisons show only that these configurations appear in different observable nighttime-light contexts. Visit volume remained the strongest standalone benchmark, with the MII and regimes adding modest descriptive information, while the São Paulo allocation demonstrated how a tract-level index can be redistributed conditionally across a finer grid without changing represented tract totals.

Together, these stages show what mobility can contribute before direct market outcomes are available, while keeping demand, opportunity, conversion, performance, and causal effects outside the claims supported by the workflow.


### 7.2 Role in market diagnosis

Visit volume is useful for locating where movement was concentrated, but places with similar volume can differ in visitor reach, recurrence, duration, and within-month stability. The MII and the exploratory regimes keep those differences visible during early market screening, while decisions about entry, location, or investment still require direct evidence about demand, competition, costs, firm capabilities, and expected performance.

The workflow is consequently a diagnostic input rather than a decision rule. It can identify local access configurations that warrant closer investigation before outcomes are available, but it cannot classify a place as profitable, underserved, or commercially successful.

### 7.3 Limitations

The mobility indicators cover August 2024 and arrive as proprietary tract-level aggregates, so the workflow compares places during one observation month without reconstructing provider-level records, following individuals, or establishing temporal persistence. The retained MII uses equal arithmetic weights and includes visit volume among its seven components, while PCA supplies an internal alignment check rather than creating or fully validating the index.

Localized signatures and regimes remain conditional on focal-location selection, neighborhood size, component scaling, and k-means. The notebook examines sensitivity to geographic weighting and neighborhood size but does not establish a unique population taxonomy of Brazilian markets.

VIIRS is available only for covered tracts and provides contextual evidence rather than a direct marketing outcome. The random training-test split does not assess transfer to geographically independent areas, while the São Paulo allocation uses VIIRS to determine its own within-tract weights and cannot treat the same surface as independent validation. Grid resolution and tract rasterization also determine which tracts are represented and how their cell values are distributed.

Within these boundaries, the workflow supports diagnostic analysis of mobility-based access conditions but does not identify causal effects, directly observe the theoretical construct, measure demand or market performance, or support entry and investment decisions without additional market evidence.


### 7.4 Reproducibility

The notebook uses project-relative paths and documents the complete analytical sequence from the analysis-ready tract layer onward. Full execution requires authorized access to the restricted mobility GeoPackage and a locally obtained annual 2024 VIIRS archive, arranged as described in `data/README.md`.

The public copy contains no mobility observations, credentials, provider-specific restricted links, device histories, individual trajectories, or personally identifiable information. Its cleared outputs and documented environment support inspection of the workflow without presenting restricted data as openly reproducible.

### 7.5 Analytical conclusion

The notebook shows how aggregated mobility can be examined as a multidimensional and spatially organized representation of local access before direct market outcomes are observed. The retained MII followed the dominant covariance structure of the seven mobility indicators, the leading local structure remained comparatively stable across weighting and neighborhood choices, and exploratory access regimes appeared in different VIIRS contexts. The Sao Paulo allocation then showed how tract-level summaries can conceal internal variation under one explicit ancillary weighting rule.

These results support a bounded use of mobility in market diagnosis: they reveal how access was organized during August 2024, not whether demand existed, whether consumers converted, or whether a firm would perform well in a location.

### 7.6 References and project materials

The dissertation chapter provides the complete theoretical framing, methodological explanation, empirical interpretation, and references. Repository documentation is available in the [Chapter 4 README](../README.md), while the dissertation-level structure is described in the [root README](../../README.md).